In [1]:
#!/usr/bin/env python3
# ============================================================================
# AI STOCK ANALYZER — PORTFOLIO-AWARE EDITION
# Multi-agent pipeline: Parallel Gemini + Claude Sonnet 4.5
# Features:
#   - 8-stage AI analysis: Research, Macro, Earnings, Insider, Trader,
#     Contrarian (Fatal Flaw), Final PM
#   - Portfolio-aware sizing: sector exposure, correlation, beta
#   - Cash management: dynamic target based on market regime + trim recs
#   - Risk scenario module: beta-adjusted stress tests
#   - Technical: OBV, A/D, Sector RS, Earnings Proximity, ATR sizing
#   - Gemini research cache: 24hr TTL to reduce repeat API costs
# ~$0.08-$0.30 per analysis | ~30-45s parallel
# ============================================================================

# --- Colab dependency install (wrapped safely) ---
try:
    import importlib
    for pkg in ["anthropic", "google-genai", "openai", "yfinance", "pandas", "matplotlib",
                "requests", "beautifulsoup4", "gradio", "python-dotenv", "edgartools"]:
        try:
            importlib.import_module(pkg.replace("-", "_").replace("google-genai", "google.genai").replace("edgartools", "edgar"))
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                          capture_output=True)
except Exception:
    pass

# --- Imports ---
import anthropic
from openai import OpenAI
from google import genai
from google.genai import types
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import gradio as gr
import base64
import io
import re
import json
import warnings
import logging
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

# --- Logging Setup ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("StockAnalyzer")

# --- API Keys ---
# Option 1 (Colab): Click the 🔑 key icon in the left sidebar, add ANTHROPIC_API_KEY
#   and GEMINI_API_KEY as secrets, and enable notebook access.
# Option 2 (VS Code): Create a .env file in the same folder as this notebook:
#   ANTHROPIC_API_KEY=your-key-here
#   GEMINI_API_KEY=your-key-here
# Option 3 (fallback): Paste your keys directly below.
ANTHROPIC_KEY_DIRECT = ""   # <- paste your Anthropic key here if needed
GEMINI_KEY_DIRECT    = ""   # <- paste your Gemini key here if needed
XAI_KEY_DIRECT       = ""   # <- paste your xAI/Grok key here if needed (optional)

from dotenv import load_dotenv
import os
load_dotenv()  # Loads .env file if running in VS Code

# Try Colab Secrets first, then .env / env vars, then direct paste
def _get_secret(name, direct_fallback):
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass  # Not in Colab, or secret not set
    return os.environ.get(name) or direct_fallback

ANTHROPIC_API_KEY  = _get_secret("ANTHROPIC_API_KEY",  ANTHROPIC_KEY_DIRECT)
GEMINI_API_KEY = _get_secret("GEMINI_API_KEY", GEMINI_KEY_DIRECT)
XAI_API_KEY    = _get_secret("XAI_API_KEY", XAI_KEY_DIRECT)  # Optional — Grok social sentiment

if not ANTHROPIC_API_KEY:
    raise ValueError("Anthropic API key missing. In Colab: add ANTHROPIC_API_KEY in the 🔑 Secrets panel. In VS Code: add to .env file or paste into ANTHROPIC_KEY_DIRECT above.")
if not GEMINI_API_KEY:
    raise ValueError("Gemini API key missing. In Colab: add GEMINI_API_KEY in the 🔑 Secrets panel. In VS Code: add to .env file or paste into GEMINI_KEY_DIRECT above.")

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
# xAI/Grok client — optional (social sentiment stage). None if no key provided.
xai_client = OpenAI(api_key=XAI_API_KEY, base_url="https://api.x.ai/v1") if XAI_API_KEY else None

In [2]:
# ============================================================================
# GEMINI RESEARCH CACHE (24-hour TTL)
# Avoids repeat API calls + grounding charges for same ticker same day
# ============================================================================

_gemini_cache = {}  # key: "TICKER:function_name:YYYYMMDD" -> value: response text

def _cache_key(ticker, func_name):
    """Generate a date-stamped cache key."""
    return f"{ticker}:{func_name}:{datetime.now().strftime('%Y%m%d')}"

def _cache_get(ticker, func_name):
    """Retrieve cached Gemini result if available and fresh (same day)."""
    key = _cache_key(ticker, func_name)
    if key in _gemini_cache:
        log.info(f"Cache HIT: {key}")
        return _gemini_cache[key]
    return None

def _cache_set(ticker, func_name, result):
    """Store Gemini result in cache."""
    key = _cache_key(ticker, func_name)
    _gemini_cache[key] = result
    # Prune old entries (different date)
    today = datetime.now().strftime('%Y%m%d')
    stale = [k for k in _gemini_cache if not k.endswith(today)]
    for k in stale:
        del _gemini_cache[k]
    log.info(f"Cache SET: {key} (cache size: {len(_gemini_cache)})")

In [3]:
# ============================================================================
# PORTFOLIO CONTEXT PARSER
# ============================================================================

def parse_portfolio(portfolio_text):
    """
    Parse portfolio input text into structured data.
    Accepts formats like:
      AAPL:15%, NVDA:10%, MSFT:12%, CASH:20%
      AAPL 15, NVDA 10, MSFT 12, CASH 20
      AAPL:15%:$50000, NVDA:10%:$33000  (with dollar amounts)
    Returns dict: {
        "holdings": {"AAPL": 15.0, "NVDA": 10.0, ...},
        "cash_pct": 20.0,
        "total_allocated": 80.0,
        "portfolio_value": None or float,
        "sectors": {},  # filled later
        "raw": "original text"
    }
    """
    if not portfolio_text or not portfolio_text.strip():
        return None

    holdings = {}
    cash_pct = 0.0
    portfolio_value = None

    # Clean and split
    text = portfolio_text.strip().upper()
    # Support both comma and newline separators
    entries = re.split(r'[,\n]+', text)

    # Cash-equivalent tickers — treated as cash in portfolio analysis
    CASH_EQUIVALENTS = {"CASH", "SGOV", "BIL", "SHV", "TBIL", "USFR"}

    # Broad index funds — counted as holdings but with reduced weight
    # in sector concentration analysis (they provide general equity exposure)
    BROAD_INDEX_FUNDS = {
        "SPY", "SPYM", "VOO", "IVV", "VTI", "ITOT", "SPTM", "SCHB",
        "VT", "VXUS", "QQQ", "DIA", "IWM", "VTV", "VUG", "RSP",
    }

    for entry in entries:
        entry = entry.strip()
        if not entry:
            continue

        # Try to parse "TICKER:PCT%" or "TICKER:PCT" or "TICKER PCT"
        match = re.match(
            r'([A-Z]{1,6})\s*[:\s]\s*(\d+(?:\.\d+)?)\s*%?'
            r'(?:\s*[:\s]\s*\$?([\d,]+(?:\.\d+)?))?',
            entry
        )
        if match:
            ticker = match.group(1)
            pct = float(match.group(2))
            if match.group(3):
                val = float(match.group(3).replace(',', ''))
                if portfolio_value is None:
                    portfolio_value = 0
                portfolio_value += val

            if ticker in CASH_EQUIVALENTS:
                cash_pct += pct
            else:
                holdings[ticker] = pct

    if not holdings and cash_pct == 0:
        return None

    total_allocated = sum(holdings.values()) + cash_pct

    return {
        "holdings": holdings,
        "cash_pct": cash_pct,
        "total_allocated": total_allocated,
        "portfolio_value": portfolio_value,
        "sectors": {},
        "raw": portfolio_text.strip(),
    }


def enrich_portfolio(portfolio):
    """
    Add sector breakdown, correlations, and beta for portfolio holdings.
    Handles ETFs by using fund category or name when sector is unavailable.
    """
    if portfolio is None:
        return None

    sectors = {}
    betas = {}
    close_data = {}

    for ticker, pct in portfolio["holdings"].items():
        try:
            stk = yf.Ticker(ticker)
            info = stk.info or {}
            quote_type = info.get("quoteType", "").upper()
            sector = info.get("sector", "")

            # For ETFs/funds: sector is empty, so use category or name
            if not sector or sector.strip() == "":
                if quote_type == "ETF":
                    category = info.get("category", "")
                    if category:
                        sector = f"ETF: {category}"
                    else:
                        # Try to infer from name
                        name = info.get("longName", info.get("shortName", ticker))
                        sector = f"ETF: {name}"
                else:
                    sector = "Unknown"

            beta = info.get("beta", info.get("beta3Year", None))

            sectors[ticker] = sector
            if beta is not None:
                betas[ticker] = float(beta)

            df = stk.history(period="6mo")
            if not df.empty and len(df) >= 30:
                close_data[ticker] = df["Close"].tail(60).pct_change().dropna().values

        except Exception as e:
            log.warning(f"Portfolio enrichment failed for {ticker}: {e}")
            sectors[ticker] = "Unknown"

    portfolio["sectors"] = sectors
    portfolio["betas"] = betas

    # Broad index funds — these provide general equity exposure and should
    # be weighted less in sector concentration analysis
    BROAD_INDEX_FUNDS = {
        "SPY", "SPYM", "VOO", "IVV", "VTI", "ITOT", "SPTM", "SCHB",
        "VT", "VXUS", "QQQ", "DIA", "IWM", "VTV", "VUG", "RSP",
    }
    INDEX_CONCENTRATION_DISCOUNT = 0.25  # count 25% of index weight toward sector

    # Track which holdings are index funds
    portfolio["index_funds"] = {t: pct for t, pct in portfolio["holdings"].items()
                                if t in BROAD_INDEX_FUNDS}

    # Sector concentration (index funds get discounted weight)
    sector_weights = {}
    for ticker, pct in portfolio["holdings"].items():
        sec = sectors.get(ticker, "Unknown")
        if not sec or sec.strip() == "":
            sec = "Unknown"
        if ticker in BROAD_INDEX_FUNDS:
            # Index funds are diversified — only a fraction counts as sector concentration
            sec = f"Broad Index ({sec})" if "ETF" in sec else "Broad Index"
            sector_weights[sec] = sector_weights.get(sec, 0) + pct * INDEX_CONCENTRATION_DISCOUNT
        else:
            sector_weights[sec] = sector_weights.get(sec, 0) + pct
    portfolio["sector_weights"] = sector_weights

    # Cross-correlations
    correlations = {}
    tickers_list = list(close_data.keys())
    for i in range(len(tickers_list)):
        for j in range(i + 1, len(tickers_list)):
            t1, t2 = tickers_list[i], tickers_list[j]
            try:
                min_len = min(len(close_data[t1]), len(close_data[t2]))
                if min_len > 10:
                    corr = np.corrcoef(
                        close_data[t1][:min_len],
                        close_data[t2][:min_len]
                    )[0, 1]
                    correlations[f"{t1}-{t2}"] = round(corr, 2)
            except Exception:
                pass
    portfolio["correlations"] = correlations

    # Weighted portfolio beta
    weighted_beta = 0
    beta_weight = 0
    for ticker, pct in portfolio["holdings"].items():
        if ticker in betas:
            weighted_beta += betas[ticker] * pct
            beta_weight += pct
    portfolio["portfolio_beta"] = round(weighted_beta / beta_weight, 2) if beta_weight > 0 else None

    return portfolio


def format_portfolio_context(portfolio, new_ticker=None, new_ticker_info=None):
    """Format portfolio data into a concise string for AI prompts."""
    if portfolio is None:
        return ""

    lines = ["CURRENT PORTFOLIO:"]

    # Holdings
    index_funds = portfolio.get("index_funds", {})
    for ticker, pct in portfolio["holdings"].items():
        sec = portfolio["sectors"].get(ticker, "?")
        beta = portfolio.get("betas", {}).get(ticker)
        beta_str = f" β={beta:.2f}" if beta else ""
        idx_tag = " [INDEX FUND — general equity exposure]" if ticker in index_funds else ""
        lines.append(f"  {ticker}: {pct:.1f}% ({sec}){beta_str}{idx_tag}")

    lines.append(f"  CASH: {portfolio['cash_pct']:.1f}%")
    lines.append(f"  Total allocated: {portfolio['total_allocated']:.1f}%")

    # Note about index fund treatment
    if index_funds:
        idx_list = ", ".join(index_funds.keys())
        lines.append(f"  NOTE: {idx_list} are broad index funds used for general equity exposure. "
                      f"They are diversified across sectors and should be weighted LESS in concentration analysis. "
                      f"SPYM/SPY-type holdings serve as long-term equity parking, not single-sector bets.")

    if portfolio.get("portfolio_beta"):
        lines.append(f"  Portfolio Beta: {portfolio['portfolio_beta']:.2f}")

    # Sector concentration
    if portfolio.get("sector_weights"):
        lines.append("SECTOR EXPOSURE:")
        for sec, wt in sorted(portfolio["sector_weights"].items(), key=lambda x: -x[1]):
            flag = " ⚠CONCENTRATED" if wt > 25 else ""
            lines.append(f"  {sec}: {wt:.1f}%{flag}")

    # Check correlation of new ticker with existing holdings
    if new_ticker and new_ticker in portfolio.get("correlations_with_new", {}):
        lines.append(f"CORRELATION WITH {new_ticker}:")
        for pair, corr in portfolio["correlations_with_new"].items():
            if abs(corr) > 0.6:
                flag = " ⚠HIGH" if abs(corr) > 0.8 else ""
                lines.append(f"  {pair}: {corr:.2f}{flag}")

    # High internal correlations
    high_corr = {k: v for k, v in portfolio.get("correlations", {}).items() if abs(v) > 0.8}
    if high_corr:
        lines.append("HIGH INTERNAL CORRELATIONS:")
        for pair, corr in high_corr.items():
            lines.append(f"  {pair}: {corr:.2f}")

    return "\n".join(lines)


def compute_new_ticker_correlations(portfolio, new_ticker):
    """Compute correlation between new ticker and each existing holding."""
    if portfolio is None:
        return portfolio

    try:
        new_df = yf.Ticker(new_ticker).history(period="6mo")
        if new_df.empty or len(new_df) < 30:
            return portfolio

        new_returns = new_df["Close"].tail(60).pct_change().dropna().values
        correlations_with_new = {}

        for ticker in portfolio["holdings"]:
            try:
                hold_df = yf.Ticker(ticker).history(period="6mo")
                if hold_df.empty or len(hold_df) < 30:
                    continue
                hold_returns = hold_df["Close"].tail(60).pct_change().dropna().values
                min_len = min(len(new_returns), len(hold_returns))
                if min_len > 10:
                    corr = np.corrcoef(new_returns[:min_len], hold_returns[:min_len])[0, 1]
                    correlations_with_new[f"{new_ticker}-{ticker}"] = round(corr, 2)
            except Exception:
                pass

        portfolio["correlations_with_new"] = correlations_with_new
    except Exception as e:
        log.warning(f"New ticker correlation failed: {e}")

    return portfolio


# ============================================================================
# RISK SCENARIO MODULE
# ============================================================================

def run_risk_scenarios(tech, portfolio, new_alloc_pct, info):
    """
    Stress-test the proposed position under different market scenarios.
    Returns formatted string for the report.
    """
    price = tech["price"]
    atr = tech["atr"]
    atr_pct = tech["atr_pct"]
    beta = info.get("beta", 1.0) if info else 1.0
    if beta is None:
        beta = 1.0

    portfolio_value_str = ""
    pv = None
    if portfolio and portfolio.get("portfolio_value"):
        pv = portfolio["portfolio_value"]
        portfolio_value_str = f" (${pv:,.0f} portfolio)"

    scenarios = []
    scenarios.append("=" * 55)
    scenarios.append(f"RISK SCENARIOS — {new_alloc_pct:.1f}% position @ ${price:.2f}{portfolio_value_str}")
    scenarios.append(f"Stock Beta: {beta:.2f} | ATR: ${atr:.2f} ({atr_pct:.1f}%/day)")
    scenarios.append("=" * 55)

    # Scenario definitions: (name, market_move%)
    market_scenarios = [
        ("Normal Day (1σ)", -1.0),
        ("Bad Week", -3.0),
        ("Correction (-10%)", -10.0),
        ("Bear Market (-20%)", -20.0),
        ("Crash (-30%)", -30.0),
    ]

    for name, market_move in market_scenarios:
        stock_move = market_move * beta
        new_price = price * (1 + stock_move / 100)
        position_loss_pct = stock_move  # loss on the position itself

        if pv:
            position_value = pv * (new_alloc_pct / 100)
            dollar_loss = position_value * (stock_move / 100)
            portfolio_impact = dollar_loss / pv * 100
            scenarios.append(
                f"  {name:.<30} Mkt {market_move:+.1f}% → Stock {stock_move:+.1f}% "
                f"→ ${new_price:.2f} | Portfolio impact: {portfolio_impact:+.2f}% (${dollar_loss:+,.0f})"
            )
        else:
            scenarios.append(
                f"  {name:.<30} Mkt {market_move:+.1f}% → Stock {stock_move:+.1f}% "
                f"→ ${new_price:.2f} | Position loss: {position_loss_pct:+.1f}%"
            )

    # Stop-loss scenario
    stop_dist = atr * 2
    stop_price = price - stop_dist
    stop_loss_pct = (stop_dist / price) * 100
    if pv:
        stop_dollar = pv * (new_alloc_pct / 100) * (stop_loss_pct / 100)
        scenarios.append(
            f"\n  Stop-loss (2×ATR).............. ${stop_price:.2f} "
            f"→ Position: -{stop_loss_pct:.1f}% | Portfolio: -{stop_dollar / pv * 100:.2f}% (${stop_dollar:,.0f})"
        )
    else:
        scenarios.append(
            f"\n  Stop-loss (2×ATR).............. ${stop_price:.2f} → Position: -{stop_loss_pct:.1f}%"
        )

    # Max pain: what if position goes to zero (for context)
    if pv:
        scenarios.append(
            f"  Max pain (total loss).......... Portfolio: -{new_alloc_pct:.1f}% "
            f"(${pv * new_alloc_pct / 100:,.0f})"
        )

    return "\n".join(scenarios)


def recommend_cash_position(fg, portfolio, tech):
    """
    Recommend target cash allocation based on market regime.
    Returns (recommended_cash_pct, reasoning, trim_suggestions).
    """
    # Base cash recommendation from Fear & Greed
    fg_score = fg.get("score") if fg else None

    if fg_score is None:
        base_cash = 15.0
        regime = "Unknown (defaulting to moderate)"
    elif fg_score <= 20:
        base_cash = 5.0
        regime = "Extreme Fear — deploy cash aggressively (contrarian)"
    elif fg_score <= 35:
        base_cash = 10.0
        regime = "Fear — cautious accumulation, keep some dry powder"
    elif fg_score <= 55:
        base_cash = 15.0
        regime = "Neutral — maintain standard cash buffer"
    elif fg_score <= 75:
        base_cash = 20.0
        regime = "Greed — raise cash, market overextended"
    else:
        base_cash = 30.0
        regime = "Extreme Greed — maximum defensiveness, high cash"

    # Adjust for portfolio beta
    if portfolio and portfolio.get("portfolio_beta"):
        pb = portfolio["portfolio_beta"]
        if pb > 1.3:
            base_cash += 5
            regime += f" | High-beta portfolio ({pb:.2f}) — extra cash buffer"
        elif pb < 0.7:
            base_cash -= 3
            regime += f" | Low-beta portfolio ({pb:.2f}) — can run leaner"

    # Adjust for sector concentration
    if portfolio and portfolio.get("sector_weights"):
        max_sector_wt = max(portfolio["sector_weights"].values())
        if max_sector_wt > 35:
            base_cash += 5
            top_sector = max(portfolio["sector_weights"], key=portfolio["sector_weights"].get)
            regime += f" | Heavy {top_sector} concentration ({max_sector_wt:.0f}%) — diversification buffer"

    recommended_cash = round(min(50, max(5, base_cash)), 0)

    # Trim suggestions if current cash is too low
    trim_suggestions = []
    if portfolio:
        current_cash = portfolio.get("cash_pct", 0)
        deficit = recommended_cash - current_cash

        if deficit > 3:  # Only suggest trims if meaningfully underweight cash
            # Sort holdings by size (largest first) and suggest trimming
            sorted_holdings = sorted(
                portfolio["holdings"].items(),
                key=lambda x: -x[1]
            )
            remaining_deficit = deficit
            for ticker, pct in sorted_holdings:
                if remaining_deficit <= 0:
                    break
                # Suggest trimming up to 30% of a position or the remaining deficit
                max_trim = min(pct * 0.3, remaining_deficit)
                if max_trim >= 1:  # Only suggest if >= 1%
                    trim_suggestions.append({
                        "ticker": ticker,
                        "current_pct": pct,
                        "trim_pct": round(max_trim, 1),
                        "new_pct": round(pct - max_trim, 1),
                    })
                    remaining_deficit -= max_trim

    return recommended_cash, regime, trim_suggestions

In [4]:
# ============================================================================
# SECTION 1: MARKET SENTIMENT — Fear & Greed Index
# ============================================================================

def _fear_greed_meta(score):
    """Convert numeric F&G score to label, emoji, and mood description."""
    s = round(score)
    if s <= 25:
        return "Extreme Fear", "😱", "Extreme Fear - contrarian buy zone"
    elif s <= 45:
        return "Fear", "😨", "Fear - cautious accumulation"
    elif s <= 55:
        return "Neutral", "😐", "Neutral"
    elif s <= 75:
        return "Greed", "😊", "Greed - watch overextension"
    else:
        return "Extreme Greed", "🤑", "Extreme Greed - pullback risk"


def get_fear_greed():
    """
    Attempt to get CNN Fear & Greed Index via:
      1. fear-and-greed pip package
      2. CNN API endpoints
      3. VIX-based proxy as fallback
    """
    # --- Method 1: pip package ---
    try:
        import subprocess, sys
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "fear-and-greed", "-q"],
            capture_output=True
        )
        import fear_and_greed
        d = fear_and_greed.get()
        s = round(float(d.value))
        label, emoji, mood = _fear_greed_meta(s)
        log.info(f"Fear & Greed via pip package: {s}")
        return {
            "score": s, "label": label, "emoji": emoji,
            "mood": mood, "prev_week": None, "trend": "N/A", "source": "CNN"
        }
    except Exception as e:
        log.warning(f"F&G pip package failed: {e}")

    # --- Method 2: CNN API ---
    urls = [
        "https://production.dataviz.cnn.io/index/fearandgreed/graphdata",
        "https://production.dataviz.cnn.io/index/fearandgreed/graphdata/current"
    ]
    headers_list = [
        {
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/json",
            "Referer": "https://www.cnn.com/markets/fear-and-greed"
        },
        {"User-Agent": "Mozilla/5.0 (Mac)", "Accept": "application/json"}
    ]
    for url in urls:
        for headers in headers_list:
            try:
                r = requests.get(url, headers=headers, timeout=8)
                if r.status_code != 200:
                    continue
                d = r.json()
                fg_data = d.get("fear_and_greed", {})
                s = round(float(fg_data.get("score", 0)))
                if s == 0:
                    continue

                label, emoji, mood = _fear_greed_meta(s)
                historical = d.get("fear_and_greed_historical", {}).get("data", [])
                prev_week = None
                if len(historical) >= 8:
                    try:
                        prev_week = round(float(historical[-8]["x"]))
                    except (KeyError, TypeError, ValueError):
                        prev_week = None

                if prev_week is not None:
                    delta = abs(s - prev_week)
                    arrow = "▲" if s > prev_week else "▼"
                    trend = f"{arrow} {delta} pts"
                else:
                    trend = "N/A"

                log.info(f"Fear & Greed via CNN API: {s}")
                return {
                    "score": s, "label": label, "emoji": emoji,
                    "mood": mood, "prev_week": prev_week,
                    "trend": trend, "source": "CNN API"
                }
            except Exception as e:
                log.warning(f"CNN API attempt failed ({url}): {e}")
                continue

    # --- Method 3: VIX proxy ---
    try:
        vx = yf.Ticker("^VIX").history(period="35d")
        sp = yf.Ticker("SPY").history(period="35d")
        if not vx.empty and not sp.empty:
            vix_now = vx["Close"].iloc[-1]
            vix_week_ago = vx["Close"].iloc[-6] if len(vx) >= 6 else vix_now
            spy_return = (
                (sp["Close"].iloc[-1] / sp["Close"].iloc[-22] - 1) * 100
                if len(sp) >= 22 else 0
            )
            # Map VIX 10-45 range to 100-0 score
            vix_score = max(0, min(100, round(100 - ((vix_now - 10) / 35) * 100)))
            # Adjust with SPY momentum (capped at +/-20)
            momentum_adj = max(-20, min(20, round(spy_return * 2)))
            proxy_score = max(0, min(100, vix_score + momentum_adj))

            label, emoji, mood = _fear_greed_meta(proxy_score)
            vix_score_week = max(0, min(100, round(100 - ((vix_week_ago - 10) / 35) * 100)))
            delta = abs(proxy_score - vix_score_week)
            arrow = "▲" if proxy_score >= vix_score_week else "▼"

            log.info(f"Fear & Greed via VIX proxy: {proxy_score} (VIX={vix_now:.1f})")
            return {
                "score": proxy_score, "label": label, "emoji": emoji,
                "mood": mood + f" (VIX:{vix_now:.1f})",
                "prev_week": vix_score_week,
                "trend": f"{arrow} {delta}", "source": "VIX proxy"
            }
    except Exception as e:
        log.warning(f"VIX proxy failed: {e}")

    log.error("All Fear & Greed methods failed.")
    return {
        "score": None, "label": "N/A", "emoji": "❓",
        "mood": "Unavailable", "prev_week": None,
        "trend": "N/A", "source": "None"
    }

In [5]:
# ============================================================================
# SECTION 2: DATA RETRIEVAL & FUNDAMENTALS
# ============================================================================

def get_stock_data(ticker):
    """Download 2 years of price history and info dict from yfinance."""
    try:
        stk = yf.Ticker(ticker)
        df = stk.history(period="2y")
        if df.empty:
            return None, None, "No data found for this ticker."
        return df, stk.info, None
    except Exception as e:
        log.error(f"get_stock_data({ticker}) failed: {e}")
        return None, None, str(e)


def is_etf(info):
    return info.get("quoteType", "").upper() == "ETF"


def _safe_fmt(value, fmt=".2f"):
    """Format a numeric value safely. Returns 'N/A' only if value is None."""
    if value is None:
        return "N/A"
    try:
        return f"{value:{fmt}}"
    except (TypeError, ValueError):
        return "N/A"


def _analyst_rating(info):
    """Parse analyst recommendation into human-readable string."""
    mean_rating = info.get("recommendationMean")
    count = info.get("numberOfAnalystOpinions")
    if mean_rating is None:
        return "N/A"
    try:
        m = float(mean_rating)
        if m <= 1.5:
            label = "Strong Buy"
        elif m <= 2.5:
            label = "Buy"
        elif m <= 3.5:
            label = "Hold"
        elif m <= 4.5:
            label = "Sell"
        else:
            label = "Strong Sell"
        if count:
            return f"{label} ({count}) [{m:.1f}/5]"
        return f"{label} [{m:.1f}/5]"
    except (TypeError, ValueError):
        return "N/A"


def get_fundamentals(info):
    """Extract key fundamental data points from yfinance info dict."""
    def cap(val, divisor=1e9, suffix="B"):
        if val is None:
            return "N/A"
        return f"${val / divisor:.2f}{suffix}"

    def pct(val):
        if val is None:
            return "N/A"
        return f"{val * 100:.1f}%"

    return {
        "Market Cap": cap(info.get("marketCap")),
        "P/E": _safe_fmt(info.get("trailingPE")),
        "Fwd P/E": _safe_fmt(info.get("forwardPE")),
        "PEG": _safe_fmt(info.get("pegRatio")),
        "P/B": _safe_fmt(info.get("priceToBook")),
        "EV/EBITDA": _safe_fmt(info.get("enterpriseToEbitda")),
        "Revenue": cap(info.get("totalRevenue")),
        "Rev Growth": pct(info.get("revenueGrowth")),
        "Gross Margin": pct(info.get("grossMargins")),
        "Net Margin": pct(info.get("profitMargins")),
        "ROE": pct(info.get("returnOnEquity")),
        "D/E": _safe_fmt(info.get("debtToEquity")),
        "FCF": cap(info.get("freeCashflow")),
        "Div Yield": f"{info.get('dividendYield', 0) * 100:.2f}%"
                     if info.get('dividendYield') is not None else "N/A",
        "52W Hi": _safe_fmt(info.get("fiftyTwoWeekHigh")),
        "52W Lo": _safe_fmt(info.get("fiftyTwoWeekLow")),
        "Target": _safe_fmt(info.get("targetMeanPrice")),
        "Target Hi": _safe_fmt(info.get("targetHighPrice")),
        "Target Lo": _safe_fmt(info.get("targetLowPrice")),
        "Analyst": _analyst_rating(info),
    }


def get_etf_info(info):
    """Extract ETF-specific data points."""
    def cap(val):
        if val is None:
            return "N/A"
        return f"${val / 1e9:.2f}B"

    def pct(val):
        if val is None:
            return "N/A"
        return f"{val * 100:.2f}%"

    return {
        "Family": info.get("fundFamily", "N/A"),
        "Category": info.get("category", "N/A"),
        "Assets": cap(info.get("totalAssets")),
        "Expense": pct(info.get("annualReportExpenseRatio")),
        "52W Hi": f"${info.get('fiftyTwoWeekHigh', 0):.2f}"
                  if info.get('fiftyTwoWeekHigh') is not None else "N/A",
        "52W Lo": f"${info.get('fiftyTwoWeekLow', 0):.2f}"
                  if info.get('fiftyTwoWeekLow') is not None else "N/A",
        "YTD": pct(info.get("ytdReturn")),
        "3Y": pct(info.get("threeYearAverageReturn")),
        "5Y": pct(info.get("fiveYearAverageReturn")),
    }


def get_earnings_proximity(ticker):
    """
    Check how many days until the next earnings report.
    Returns (days_until, date_str) or (None, 'Unknown').
    """
    try:
        stk = yf.Ticker(ticker)
        cal = stk.calendar
        if cal is not None and not cal.empty:
            # yfinance returns earnings date(s) — take the earliest upcoming
            if isinstance(cal, pd.DataFrame):
                if "Earnings Date" in cal.columns:
                    earn_date = pd.to_datetime(cal["Earnings Date"].iloc[0])
                elif "Earnings Date" in cal.index:
                    earn_date = pd.to_datetime(cal.loc["Earnings Date"].iloc[0])
                else:
                    return None, "Unknown"
            else:
                return None, "Unknown"

            today = pd.Timestamp.now().normalize()
            days_until = (earn_date - today).days
            return days_until, earn_date.strftime("%b %d, %Y")
    except Exception as e:
        log.warning(f"Earnings proximity check failed for {ticker}: {e}")

    return None, "Unknown"


# ============================================================================

In [6]:
# ============================================================================
# SECTION 2B: SEC FILING DATA (edgartools)
# Pulls structured financials (XBRL) + narrative sections from 10-K/10-Q
# Supplements yfinance with deeper data — skipped for ETFs
# No API key required — talks directly to SEC EDGAR
# ============================================================================

try:
    import importlib
    importlib.import_module("edgar")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "edgartools", "-q"],
                   capture_output=True)

from edgar import Company, set_identity, MultiFinancials  # type: ignore
set_identity("stockanalyzer@example.com")  # Required by SEC — any valid email

# Cache for SEC data (same pattern as Gemini cache)
_sec_cache = {}


def _sec_cache_key(ticker):
    """Date-stamped cache key for SEC data."""
    return f"{ticker}:sec:{datetime.now().strftime('%Y%m%d')}"


def get_sec_filing_data(ticker, etf=False):
    """
    Pull structured financial data + narrative sections from SEC filings.
    Returns a dict with:
      - financials_text: formatted multi-year financials for AI prompts
      - mda_text: Management Discussion & Analysis (truncated)
      - risk_factors_text: Risk Factors section (truncated)
      - filing_date: date of the filing used
      - source: "10-K" or "10-Q"
    Returns None for ETFs or if data is unavailable.
    """
    if etf:
        log.info(f"SEC filing skip: {ticker} is an ETF")
        return None

    # Check cache
    cache_key = _sec_cache_key(ticker)
    if cache_key in _sec_cache:
        log.info(f"SEC cache HIT: {ticker}")
        return _sec_cache[cache_key]

    try:
        company = Company(ticker)
        result = {
            "financials_text": "",
            "mda_text": "",
            "risk_factors_text": "",
            "filing_date": "Unknown",
            "source": "N/A",
        }

        # --- 1. Multi-year structured financials (XBRL) ---
        # Try to get last 3 years of 10-K data for trend analysis
        financials_text = _extract_multi_year_financials(company)
        if financials_text:
            result["financials_text"] = financials_text

        # --- 2. Narrative sections from most recent 10-K ---
        # These contain the real qualitative insights
        mda, risks, filing_date, source = _extract_narrative_sections(company)
        result["mda_text"] = mda
        result["risk_factors_text"] = risks
        result["filing_date"] = filing_date
        result["source"] = source

        # Cache the result
        _sec_cache[cache_key] = result
        # Prune stale cache entries
        today = datetime.now().strftime('%Y%m%d')
        stale = [k for k in _sec_cache if not k.endswith(today)]
        for k in stale:
            del _sec_cache[k]

        log.info(f"SEC data loaded for {ticker}: {source}, dated {filing_date}, "
                 f"financials={len(result['financials_text'])}ch, "
                 f"mda={len(result['mda_text'])}ch, risks={len(result['risk_factors_text'])}ch")
        return result

    except Exception as e:
        log.warning(f"SEC filing data unavailable for {ticker}: {e}")
        return None


def _extract_multi_year_financials(company):
    """
    Extract 3 years of key financial metrics from XBRL filings.
    Returns a compact, prompt-friendly string.
    """
    try:
        filings_10k = company.get_filings(form="10-K")
        # Get up to 3 most recent 10-K filings
        recent_10ks = filings_10k.head(3) if hasattr(filings_10k, 'head') else filings_10k[:3]

        if not recent_10ks or len(recent_10ks) == 0:
            return ""

        multi = MultiFinancials.extract(recent_10ks)

        lines = ["SEC FINANCIAL STATEMENTS (XBRL — last 3 years):"]

        # Income statement
        try:
            income = multi.income_statement()
            if income is not None:
                lines.append(f"\nINCOME STATEMENT:\n{str(income)[:2000]}")
        except Exception as e:
            log.debug(f"Income statement extraction failed: {e}")

        # Balance sheet
        try:
            balance = multi.balance_sheet()
            if balance is not None:
                lines.append(f"\nBALANCE SHEET:\n{str(balance)[:1500]}")
        except Exception as e:
            log.debug(f"Balance sheet extraction failed: {e}")

        # Cash flow
        try:
            cashflow = multi.cashflow_statement()
            if cashflow is not None:
                lines.append(f"\nCASH FLOW:\n{str(cashflow)[:1500]}")
        except Exception as e:
            log.debug(f"Cash flow extraction failed: {e}")

        if len(lines) > 1:
            return "\n".join(lines)
        return ""

    except Exception as e:
        log.debug(f"Multi-year financials failed: {e}")
        return ""


def _extract_narrative_sections(company):
    """
    Extract MD&A and Risk Factors from the most recent 10-K or 10-Q.
    Returns (mda_text, risk_factors_text, filing_date, source_type).
    Truncates to keep prompt size manageable.
    """
    MAX_MDA_CHARS = 4000       # ~1000 tokens — enough for key themes
    MAX_RISK_CHARS = 3000      # ~750 tokens — top risks only

    mda = ""
    risks = ""
    filing_date = "Unknown"
    source = "N/A"

    # Try 10-K first (annual — more comprehensive)
    try:
        filings_10k = company.get_filings(form="10-K")
        latest_10k = filings_10k.latest()
        if latest_10k:
            tenk = latest_10k.obj()  # Parse into TenK object
            filing_date = str(latest_10k.filing_date) if hasattr(latest_10k, 'filing_date') else "Unknown"
            source = "10-K"

            # MD&A (Item 7)
            try:
                mda_raw = tenk.management_discussion
                if mda_raw:
                    mda_text = str(mda_raw)
                    # Intelligent truncation: keep the opening (usually has the summary)
                    if len(mda_text) > MAX_MDA_CHARS:
                        mda = mda_text[:MAX_MDA_CHARS] + "\n[...truncated — full MD&A in SEC filing]"
                    else:
                        mda = mda_text
            except Exception as e:
                log.debug(f"MD&A extraction failed: {e}")

            # Risk Factors (Item 1A)
            try:
                risk_raw = tenk.risk_factors
                if risk_raw:
                    risk_text = str(risk_raw)
                    if len(risk_text) > MAX_RISK_CHARS:
                        risks = risk_text[:MAX_RISK_CHARS] + "\n[...truncated — full risk factors in SEC filing]"
                    else:
                        risks = risk_text
            except Exception as e:
                log.debug(f"Risk factors extraction failed: {e}")

            if mda or risks:
                return mda, risks, filing_date, source

    except Exception as e:
        log.debug(f"10-K narrative extraction failed: {e}")

    # Fallback: try most recent 10-Q
    try:
        filings_10q = company.get_filings(form="10-Q")
        latest_10q = filings_10q.latest()
        if latest_10q:
            tenq = latest_10q.obj()
            filing_date = str(latest_10q.filing_date) if hasattr(latest_10q, 'filing_date') else "Unknown"
            source = "10-Q"

            # 10-Q sections are structured differently — try items
            try:
                sections = tenq.sections
                if sections:
                    # Look for MD&A-like section
                    sec_text = str(sections)
                    if len(sec_text) > MAX_MDA_CHARS:
                        mda = sec_text[:MAX_MDA_CHARS] + "\n[...truncated]"
                    else:
                        mda = sec_text
            except Exception as e:
                log.debug(f"10-Q section extraction failed: {e}")

            return mda, risks, filing_date, source

    except Exception as e:
        log.debug(f"10-Q fallback failed: {e}")

    return mda, risks, filing_date, source


def format_sec_context(sec_data):
    """
    Format SEC filing data into a compact string for AI prompts.
    Designed to complement (not duplicate) the yfinance fundamentals.
    """
    if sec_data is None:
        return ""

    lines = []

    if sec_data.get("financials_text"):
        lines.append(sec_data["financials_text"])

    if sec_data.get("mda_text"):
        lines.append(f"\nMD&A ({sec_data['source']}, filed {sec_data['filing_date']}):")
        lines.append(sec_data["mda_text"])

    if sec_data.get("risk_factors_text"):
        lines.append(f"\nKEY RISK FACTORS ({sec_data['source']}):")
        lines.append(sec_data["risk_factors_text"])

    return "\n".join(lines)


def get_enhanced_fundamentals(info, sec_data=None, etf=False):
    """
    Merge yfinance snapshot metrics with SEC XBRL data.
    yfinance provides: real-time price metrics, analyst targets, ratios
    SEC provides: multi-year trends, detailed line items, as-reported data

    This avoids redundancy: yfinance handles what it's good at (live data),
    SEC handles what yfinance can't (historical statements, filing narratives).
    """
    if etf:
        # ETFs don't file 10-K/10-Q — use yfinance only
        return get_etf_info(info), ""

    # Start with yfinance snapshot (real-time, analyst data)
    fd = get_fundamentals(info)

    # Build the SEC deep-dive string for AI prompts
    sec_context = ""
    if sec_data:
        sec_context = format_sec_context(sec_data)

    return fd, sec_context


01:16:15 [INFO] Identity of the Edgar REST client set to [stockanalyzer@example.com]


In [7]:
# SECTION 3: TECHNICAL INDICATORS
# ============================================================================

def calculate_indicators(df):
    """Calculate all technical indicators on the price DataFrame."""
    close = df["Close"]
    high = df["High"]
    low = df["Low"]
    volume = df["Volume"]

    # --- Moving Averages ---
    df["MA20"] = close.rolling(20).mean()
    df["MA50"] = close.rolling(50).mean()
    df["MA200"] = close.rolling(200).mean()

    # --- RSI (with epsilon to prevent division by zero) ---
    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df["RSI"] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # --- ATR (Average True Range) ---
    hl = high - low
    hc = np.abs(high - close.shift())
    lc = np.abs(low - close.shift())
    df["ATR"] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(14).mean()

    # --- MACD ---
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["Hist"] = df["MACD"] - df["Signal"]

    # --- Bollinger Bands ---
    df["BB_Mid"] = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    df["BB_Upper"] = df["BB_Mid"] + 2 * bb_std
    df["BB_Lower"] = df["BB_Mid"] - 2 * bb_std

    # --- Volume MA ---
    df["Vol_MA20"] = volume.rolling(20).mean()

    # --- Stochastic Oscillator ---
    low14 = low.rolling(14).min()
    high14 = high.rolling(14).max()
    df["Stoch_K"] = 100 * (close - low14) / (high14 - low14 + 1e-9)
    df["Stoch_D"] = df["Stoch_K"].rolling(3).mean()

    # --- VWAP (rolling 20-day) ---
    tp = (high + low + close) / 3
    df["VWAP"] = (tp * volume).rolling(20).sum() / (volume.rolling(20).sum() + 1e-9)

    # --- NEW: On-Balance Volume (OBV) ---
    obv_sign = np.where(close > close.shift(1), 1, np.where(close < close.shift(1), -1, 0))
    df["OBV"] = (volume * obv_sign).cumsum()
    df["OBV_MA20"] = df["OBV"].rolling(20).mean()

    # --- NEW: Accumulation/Distribution Line ---
    mfm = ((close - low) - (high - close)) / (high - low + 1e-9)  # Money Flow Multiplier
    df["AD"] = (mfm * volume).cumsum()
    df["AD_MA20"] = df["AD"].rolling(20).mean()

    return df


def get_support_resistance(df, lookback=252, n=3):
    """Find support/resistance levels using local minima/maxima with improved clustering."""
    prices = df["Close"].tail(lookback).values
    highs = df["High"].tail(lookback).values
    lows = df["Low"].tail(lookback).values
    current = prices[-1]

    resistance_candidates = []
    support_candidates = []
    window = 10

    for i in range(window, len(prices) - window):
        if highs[i] == max(highs[i - window:i + window]):
            resistance_candidates.append(highs[i])
        if lows[i] == min(lows[i - window:i + window]):
            support_candidates.append(lows[i])

    supports = sorted([x for x in support_candidates if x < current], reverse=True)[:n * 2]
    resistances = sorted([x for x in resistance_candidates if x > current])[:n * 2]

    def cluster(levels, min_gap_pct=0.015):
        """Cluster nearby levels, checking against ALL accepted levels (not just the last)."""
        clustered = []
        for v in levels:
            too_close = False
            for existing in clustered:
                if abs(v - existing) / existing < min_gap_pct:
                    too_close = True
                    break
            if not too_close:
                clustered.append(v)
            if len(clustered) >= n:
                break
        return clustered

    return cluster(supports), cluster(resistances)


def get_fibonacci_levels(df, lookback=126):
    """Calculate Fibonacci retracement levels from recent high/low."""
    recent = df.tail(lookback)
    hi = recent["High"].max()
    lo = recent["Low"].min()
    diff = hi - lo
    levels = {
        f"Fib {int(x * 100)}%": round(hi - diff * x, 2)
        for x in [0, 0.236, 0.382, 0.5, 0.618, 0.786, 1]
    }
    return levels, hi, lo


def get_relative_sector_strength(ticker, info, df):
    """
    NEW: Compare stock momentum to its sector ETF over multiple timeframes.
    Returns a dict of relative strength metrics.
    """
    sector_etf_map = {
        "Technology": "XLK", "Financial Services": "XLF", "Healthcare": "XLV",
        "Energy": "XLE", "Consumer Cyclical": "XLY", "Consumer Defensive": "XLP",
        "Industrials": "XLI", "Basic Materials": "XLB", "Real Estate": "XLRE",
        "Utilities": "XLU", "Communication Services": "XLC",
    }
    sector = info.get("sector", "")
    etf_ticker = sector_etf_map.get(sector)

    if not etf_ticker:
        return {"sector_etf": "N/A", "rs_20d": "N/A", "rs_60d": "N/A",
                "rs_120d": "N/A", "verdict": "N/A (sector ETF unknown)"}

    try:
        etf_df = yf.Ticker(etf_ticker).history(period="2y")
        if etf_df.empty:
            raise ValueError("Empty ETF data")

        result = {"sector_etf": etf_ticker}
        stock_close = df["Close"]
        etf_close = etf_df["Close"]

        for days, label in [(20, "rs_20d"), (60, "rs_60d"), (120, "rs_120d")]:
            if len(stock_close) > days and len(etf_close) > days:
                stock_ret = (stock_close.iloc[-1] / stock_close.iloc[-days] - 1) * 100
                etf_ret = (etf_close.iloc[-1] / etf_close.iloc[-days] - 1) * 100
                result[label] = f"{stock_ret - etf_ret:+.1f}%"
            else:
                result[label] = "N/A"

        # Simple verdict
        try:
            rs_vals = []
            for k in ["rs_20d", "rs_60d", "rs_120d"]:
                if result[k] != "N/A":
                    rs_vals.append(float(result[k].replace("%", "").replace("+", "")))
            if rs_vals:
                avg_rs = sum(rs_vals) / len(rs_vals)
                if avg_rs > 5:
                    result["verdict"] = f"Strong outperformer vs {etf_ticker}"
                elif avg_rs > 0:
                    result["verdict"] = f"Slight outperformer vs {etf_ticker}"
                elif avg_rs > -5:
                    result["verdict"] = f"Slight underperformer vs {etf_ticker}"
                else:
                    result["verdict"] = f"Significant underperformer vs {etf_ticker}"
            else:
                result["verdict"] = "Insufficient data"
        except Exception:
            result["verdict"] = "N/A"

        return result
    except Exception as e:
        log.warning(f"Relative strength calc failed: {e}")
        return {"sector_etf": etf_ticker, "rs_20d": "N/A", "rs_60d": "N/A",
                "rs_120d": "N/A", "verdict": "Data unavailable"}


# ============================================================================

In [8]:
# SECTION 4: BACKTESTING
# ============================================================================

def run_backtest(df, tech):
    """Run signal-based backtests over 2yr of data."""
    results = []
    close = df["Close"].values
    n = len(close)
    holding_periods = [5, 10, 20, 60]

    def backtest_signal(name, signals):
        if not signals:
            return f"{name}: No signals in 2yr"
        sdf = pd.DataFrame(signals).groupby("days")["return"].agg(["mean", "count"]).reset_index()
        lines = [f"  {int(row['days'])}D: avg {row['mean']:+.1f}% ({int(row['count'])} sig)"
                 for _, row in sdf.iterrows()]
        return f"{name}:\n" + "\n".join(lines)

    # --- Golden/Death Cross ---
    ma50 = df["MA50"].values
    ma200 = df["MA200"].values
    golden_cross_signals = []
    death_cross_signals = []

    for i in range(201, n):
        if ma50[i] > ma200[i] and ma50[i - 1] <= ma200[i - 1]:
            for hp in holding_periods:
                if i + hp < n:
                    golden_cross_signals.append({
                        "days": hp,
                        "return": (close[i + hp] / close[i] - 1) * 100
                    })
        if ma50[i] < ma200[i] and ma50[i - 1] >= ma200[i - 1]:
            for hp in holding_periods:
                if i + hp < n:
                    death_cross_signals.append({
                        "days": hp,
                        "return": (close[i + hp] / close[i] - 1) * 100
                    })

    results.append(backtest_signal("GOLDEN CROSS", golden_cross_signals))
    results.append(backtest_signal("DEATH CROSS", death_cross_signals))

    # --- RSI Oversold/Overbought ---
    rsi = df["RSI"].values
    rsi_buy_signals = []
    rsi_sell_signals = []

    for i in range(15, n):
        if rsi[i] < 30 and rsi[i - 1] >= 30:
            for hp in holding_periods:
                if i + hp < n:
                    rsi_buy_signals.append({
                        "days": hp,
                        "return": (close[i + hp] / close[i] - 1) * 100
                    })
        if rsi[i] > 70 and rsi[i - 1] <= 70:
            for hp in holding_periods:
                if i + hp < n:
                    rsi_sell_signals.append({
                        "days": hp,
                        "return": (close[i + hp] / close[i] - 1) * 100
                    })

    results.append(backtest_signal("RSI<30 BUY", rsi_buy_signals))
    results.append(backtest_signal("RSI>70", rsi_sell_signals))

    # --- MACD Bullish Cross ---
    macd = df["MACD"].values
    signal = df["Signal"].values
    macd_signals = []

    for i in range(27, n):
        if macd[i] > signal[i] and macd[i - 1] <= signal[i - 1]:
            for hp in holding_periods:
                if i + hp < n:
                    macd_signals.append({
                        "days": hp,
                        "return": (close[i + hp] / close[i] - 1) * 100
                    })

    results.append(backtest_signal("MACD BULL CROSS", macd_signals))

    # --- Bollinger Band Lower Touch ---
    bb_lower = df["BB_Lower"].values
    bb_signals = []

    for i in range(21, n):
        if close[i] <= bb_lower[i] and close[i - 1] > bb_lower[i - 1]:
            for hp in holding_periods:
                if i + hp < n:
                    bb_signals.append({
                        "days": hp,
                        "return": (close[i + hp] / close[i] - 1) * 100
                    })

    results.append(backtest_signal("BB LOWER TOUCH", bb_signals))

    # --- Buy & Hold benchmarks ---
    if n >= 252:
        results.append(f"BUY&HOLD 1Y: {(close[-1] / close[-252] - 1) * 100:+.1f}%")
    if n >= 126:
        results.append(f"BUY&HOLD 6M: {(close[-1] / close[-126] - 1) * 100:+.1f}%")

    return "\n\n".join(results)


# ============================================================================

In [9]:
# SECTION 5: CHARTING
# ============================================================================

def plot_charts(df, ticker):
    """Create a 4-panel technical chart (Price+BB, Volume, RSI+Stoch, MACD)."""
    dp = df.tail(252).copy()
    sups, ress = get_support_resistance(df)
    fibs, fh, fl = get_fibonacci_levels(df)

    fig = plt.figure(figsize=(16, 18))
    fig.patch.set_facecolor("white")
    gs = gridspec.GridSpec(5, 1, height_ratios=[3, 1, 1.2, 1, 0.8], hspace=0.08)

    # Color palette
    CLR_LINE = "#111"
    CLR_GRID = "#e0e0e0"
    CLR_BLUE = "#1a6fba"
    CLR_GREEN = "#1a7a3a"
    CLR_RED = "#cc2222"
    CLR_GOLD = "#b8860b"
    CLR_BB = "#9ecae1"
    CLR_PURPLE = "#7b2d8b"
    CLR_FIB = "#e67e00"

    # --- Panel 1: Price, MAs, BBands, S/R, Fibs ---
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor("#fafafa")
    ax1.fill_between(dp.index, dp["BB_Upper"], dp["BB_Lower"], alpha=0.1, color=CLR_BB)
    ax1.plot(dp.index, dp["BB_Upper"], color=CLR_BB, lw=0.8)
    ax1.plot(dp.index, dp["BB_Lower"], color=CLR_BB, lw=0.8)
    ax1.plot(dp.index, dp["Close"], color="#222", lw=1.6, label="Price", zorder=5)
    ax1.plot(dp.index, dp["MA20"], color=CLR_GOLD, lw=1, label="MA20", ls="--")
    ax1.plot(dp.index, dp["MA50"], color=CLR_BLUE, lw=1.3, label="MA50")
    ax1.plot(dp.index, dp["MA200"], color=CLR_RED, lw=1.3, label="MA200")
    ax1.plot(dp.index, dp["VWAP"], color=CLR_PURPLE, lw=1.2, label="VWAP", ls="-.")

    for k in ["Fib 23%", "Fib 38%", "Fib 50%", "Fib 61%"]:
        if k in fibs:
            ax1.axhline(fibs[k], color=CLR_FIB, lw=0.7, ls=":", alpha=0.7)
    for s in sups:
        ax1.axhline(s, color=CLR_GREEN, lw=0.9, ls="--", alpha=0.5)
    for r in ress:
        ax1.axhline(r, color=CLR_RED, lw=0.9, ls="--", alpha=0.5)

    ax1.set_title(f"{ticker} — Technical Chart", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=7, ncol=2)
    ax1.grid(color=CLR_GRID, lw=0.6)

    # --- Panel 2: Volume ---
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    vol_colors = [CLR_GREEN if c >= o else CLR_RED for c, o in zip(dp["Close"], dp["Open"])]
    ax2.bar(dp.index, dp["Volume"], color=vol_colors, alpha=0.6, width=1)
    ax2.plot(dp.index, dp["Vol_MA20"], color=CLR_GOLD, lw=1.2)
    ax2.set_ylabel("Volume", fontsize=8)
    ax2.grid(color=CLR_GRID, lw=0.6)

    # --- Panel 3: RSI + Stochastic ---
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax3.plot(dp.index, dp["RSI"], color=CLR_BLUE, lw=1.3, label="RSI")
    ax3.axhline(70, color=CLR_RED, ls="--")
    ax3.axhline(30, color=CLR_GREEN, ls="--")
    ax3.fill_between(dp.index, dp["RSI"], 70, where=dp["RSI"] >= 70, alpha=0.1, color=CLR_RED)
    ax3.fill_between(dp.index, dp["RSI"], 30, where=dp["RSI"] <= 30, alpha=0.1, color=CLR_GREEN)
    ax3b = ax3.twinx()
    ax3b.plot(dp.index, dp["Stoch_K"], color=CLR_PURPLE, lw=1, alpha=0.8, label="Stoch %K")
    ax3b.set_ylim(0, 100)
    ax3.set_ylim(0, 100)
    ax3.set_ylabel("RSI", fontsize=8)
    ax3.grid(color=CLR_GRID, lw=0.6)

    # --- Panel 4: MACD ---
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    ax4.plot(dp.index, dp["MACD"], color=CLR_BLUE, lw=1.3, label="MACD")
    ax4.plot(dp.index, dp["Signal"], color=CLR_GOLD, lw=1.1, label="Signal")
    hist_colors = [CLR_GREEN if v >= 0 else CLR_RED for v in dp["Hist"]]
    ax4.bar(dp.index, dp["Hist"], color=hist_colors, alpha=0.5, width=1)
    ax4.set_ylabel("MACD", fontsize=8)
    ax4.grid(color=CLR_GRID, lw=0.6)

    # --- Panel 5 (NEW): OBV ---
    ax5 = fig.add_subplot(gs[4], sharex=ax1)
    ax5.plot(dp.index, dp["OBV"], color=CLR_PURPLE, lw=1.2, label="OBV")
    ax5.plot(dp.index, dp["OBV_MA20"], color=CLR_GOLD, lw=1, ls="--", label="OBV MA20")
    ax5.set_ylabel("OBV", fontsize=8)
    ax5.legend(fontsize=7)
    ax5.grid(color=CLR_GRID, lw=0.6)

    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig


def chart_to_b64(fig):
    """Convert matplotlib figure to base64 PNG string."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight", facecolor="white")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return b64


# ============================================================================

In [10]:
# SECTION 6: AI AGENT CALLS (Gemini + Claude)
# ============================================================================

def gcall(prompt, ticker=None, func_name=None):
    """Call Gemini 2.5 Flash with Google Search grounding. Uses cache if available."""
    # Check cache first
    if ticker and func_name:
        cached = _cache_get(ticker, func_name)
        if cached:
            return cached

    try:
        r = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())],
                temperature=0.3
            )
        )
        result = r.text

        # Store in cache
        if ticker and func_name:
            _cache_set(ticker, func_name, result)

        return result
    except Exception as e:
        log.error(f"Gemini call failed: {e}")
        return f"Gemini unavailable: {e}"


def gemini_research(tk, nm, fd, etf, sec_context=""):
    """Stage 2: Equity Research Analyst (Gemini + Search). Cached."""
    f = "\n".join([f"- {k}: {v}" for k, v in fd.items()])
    sec_prompt = ""
    if sec_context:
        sec_prompt = f"\nSEC FILING DATA (use this for deeper financial analysis — multi-year trends, balance sheet details, management commentary):\n{sec_context[:2500]}"
    return gcall(f"""Senior equity research analyst. Analyze {nm} ({tk}) - {'ETF' if etf else 'Stock'}.
USE GOOGLE SEARCH to find the latest information. Do not rely on old data.
Current financials: {f}

Provide a thorough research report:
1. RECENT NEWS & CATALYSTS: Search for news in the past 30 days. What is moving this stock? Earnings surprises, product launches, management changes, lawsuits, FDA approvals, partnerships, guidance changes, analyst upgrades or downgrades.
2. COMPETITIVE POSITION: How does this company compare to its top 3 competitors? Search for each competitor's recent earnings, revenue growth, P/E ratio, and stock performance. Who is gaining or losing market share? What is the competitive moat?
3. SECTOR CONTEXT: How is the broader sector performing? Search for sector ETF performance, sector rotation trends, and whether money is flowing in or out.
4. QUANTITATIVE ANALYSIS: Is the P/E ratio justified given growth rates? Compare to sector median. Price vs 52-week range. Cash flow health and debt assessment. Analyst consensus target vs current price — upside/downside percent.
{sec_prompt}
5. TOP 3 RISKS: Be specific with real, searchable risks. Not generic.
6. PRELIMINARY VERDICT: BUY, SELL, or HOLD with confidence 1-10 and one-paragraph justification. Cite actual numbers from your search.""", ticker=tk, func_name="research")


def gemini_macro(tk, nm, sector, etf):
    """Stage 3: Macro Strategist (Gemini + Search)."""
    return gcall(f"""Macro strategist analyzing {nm} ({tk}) in the {sector} sector. USE GOOGLE SEARCH.
1. INTEREST RATES: What is current Fed policy? How do interest rate expectations specifically impact {sector}? Search for latest Fed minutes or statements.
2. SECTOR ROTATION: Is money flowing into or out of {sector}? Search for sector ETF flows (XLK, XLF, XLE etc). Which sectors are gaining at the expense of others?
3. GEOPOLITICAL: Are there trade wars, tariffs, sanctions, or regulations that specifically affect {nm}? Search for recent policy changes.
4. ECONOMIC CYCLE: Where are we in the business cycle? How does {sector} typically perform at this stage? Search for leading indicators.
5. CURRENCY & COMMODITY: If relevant to {nm}, what are FX or commodity trends doing? Search for USD, oil, copper, etc.
6. MACRO VERDICT: Is the macro environment a tailwind or headwind for this stock? Rate 1-10 (10 = strong tailwind). Be specific to THIS stock, not generic.""", ticker=tk, func_name="macro")


def gemini_earnings(tk, nm, etf):
    """Stage 4: Earnings Analyst (Gemini + Search)."""
    if etf:
        return "ETF - earnings not applicable."
    return gcall(f"""Earnings analyst for {nm} ({tk}). USE GOOGLE SEARCH for all data points.
1. LAST EARNINGS: Search for the most recent earnings report. What was the date? Did they beat or miss on revenue and EPS? By how much? What was forward guidance? Was management tone optimistic or cautious? Any notable quotes?
2. ESTIMATE REVISIONS (past 30 days): Have analysts revised EPS or revenue estimates up or down? Search for recent analyst notes. Any upgrades or downgrades?
3. NEXT EARNINGS: When is the next earnings date? What is the consensus EPS and revenue estimate? What is the whisper number if available?
4. EARNINGS TREND: How many consecutive quarters of beats or misses? Is the beat magnitude growing or shrinking? Is revenue growth accelerating or decelerating?
5. EARNINGS VERDICT: Is earnings momentum POSITIVE, NEGATIVE, or NEUTRAL? Should an investor buy before or after the next report? Give specific reasoning with numbers and dates.""", ticker=tk, func_name="earnings")


def gemini_insider(tk, nm, etf):
    """Stage 5: Institutional Flow Analyst (Gemini + Search)."""
    if etf:
        return "ETF - insider tracking not applicable."
    return gcall(f"""Institutional flow analyst for {nm} ({tk}). USE GOOGLE SEARCH for all data.
1. INSIDER TRANSACTIONS (past 90 days): Search for insider buys and sells. Who (CEO, CFO, directors)? How much dollar value? Is there a pattern — cluster buying or selling? Any Form 4 filings?
2. INSTITUTIONAL OWNERSHIP: Search for recent 13F filings. Are major funds (Berkshire, BlackRock, Vanguard, ARK, etc.) adding or trimming? What is the institutional ownership percentage trend?
3. SHORT INTEREST: What is short interest as percent of float? Is it rising or falling? What is days to cover? Any short squeeze potential?
4. FUND FLOWS: For the sector, are ETFs seeing inflows or outflows? Is smart money rotating in or out?
5. SMART MONEY VERDICT: Based on insider and institutional behavior, rate BULLISH, BEARISH, or NEUTRAL on a 1-10 scale. What are the most actionable signals?""", ticker=tk, func_name="insider")



def grok_social_sentiment(ticker, company_name, etf=False):
    """Stage 5b: Social Sentiment via Grok (xAI). Uses live X/Twitter data."""
    # Return placeholder if no xAI key configured
    if not xai_client:
        return "Grok sentiment unavailable — no API key configured."

    # Check cache first (same 24hr cache as Gemini calls)
    cached = _cache_get(ticker, "social_sentiment")
    if cached:
        return cached

    try:
        if etf:
            subject = f"{company_name} ({ticker}) ETF — focus on the sector/index it tracks"
            prompt_extra = "Since this is an ETF, analyze sentiment around the sector/index it tracks, not a specific company. Look at flows, rotation sentiment, and macro narratives on X."
        else:
            subject = f"{company_name} ({ticker})"
            prompt_extra = ""

        prompt = f"""You have access to live X/Twitter data. Analyze current social sentiment for {subject}.

1) X/TWITTER SENTIMENT SNAPSHOT: Overall retail sentiment right now — bullish, bearish, or mixed? Volume of chatter — trending or quiet? Approximate post volume.
2) KEY NARRATIVES: Top 2-3 themes/narratives being discussed (e.g., earnings reaction, short squeeze, sector rotation, product launch, insider selling, FDA, legal issues).
3) FINTWIT ACTIVITY: Are notable finance accounts or traders posting about this? What is the tone?
4) SENTIMENT DIVERGENCE: Does social sentiment agree or disagree with institutional/analyst consensus? Flag any divergence — that's where alpha often hides.
5) CAUTION FLAGS: Any signs of pump-and-dump, coordinated campaigns, bot activity, or meme stock energy? Rate the manipulation risk.
6) SOCIAL SENTIMENT SCORE: Rate 1-10 (10 = extremely bullish). Note if this is contrarian to fundamentals.
{prompt_extra}
Be concise. Use actual data from X. If data is limited, say so — do not fabricate."""

        r = xai_client.chat.completions.create(
            model="grok-3-mini-fast",
            max_tokens=1500,
            temperature=0.3,
            messages=[{"role": "user", "content": prompt}]
        )
        result = r.choices[0].message.content

        # Cache the result
        _cache_set(ticker, "social_sentiment", result)
        return result

    except Exception as e:
        log.warning(f"Grok sentiment call failed for {ticker}: {e}")
        return f"Grok sentiment unavailable: {e}"

def claude_trader(tk, nm, tech, fd, research, macro, earnings, insider, fg, bt, etf, rs=None, portfolio_ctx="", sec_context="", sentiment=""):
    """Stage 6: Senior Trader (Claude Sonnet). Now portfolio-aware."""
    f = "\n".join([f"- {k}: {v}" for k, v in fd.items()])
    sec_prompt = ""
    if sec_context:
        sec_prompt = f"\nSEC FILING DATA (use this for deeper financial analysis — multi-year trends, balance sheet details, management commentary):\n{sec_context[:2500]}"
    sup = ", ".join([f"${s:.2f}" for s in tech["supports"]]) or "N/A"
    res = ", ".join([f"${r:.2f}" for r in tech["resistances"]]) or "N/A"
    fib = "\n".join([
        f"  {k}: ${v:.2f}"
        for k, v in list(tech["fib_levels"].items())[1:-1]
    ])
    fgs = (f"F&G: {fg['score']}/100 ({fg['label']}) {fg['trend']}"
           if fg and fg["score"] else "F&G: N/A")

    # Include OBV and relative strength data
    obv_info = f"OBV trend: {'Bullish' if tech.get('obv_bullish') else 'Bearish/Flat'}"
    rs_info = ""
    if rs and rs.get("verdict", "N/A") != "N/A":
        rs_info = f"\nSECTOR RS: {rs['verdict']} (20d:{rs.get('rs_20d','N/A')}, 60d:{rs.get('rs_60d','N/A')}, 120d:{rs.get('rs_120d','N/A')})"

    portfolio_section = ""
    if portfolio_ctx:
        portfolio_section = f"\n\n{portfolio_ctx}\n\nIMPORTANT: Size the position considering existing portfolio exposure. Flag if adding this stock creates sector concentration >30% or high correlation with existing holdings."

    # SEC filing context — multi-year financials + MD&A narrative
    sec_section = ""
    if sec_context:
        sec_section = f"\nSEC FILINGS:\n{sec_context[:3000]}"

    prompt = f"""Senior TRADER. Full intel below.
TECH: ${tech['price']:.2f} MA20=${tech['ma20']:.2f} MA50=${tech['ma50']:.2f} MA200=${tech['ma200']:.2f} {"Golden" if tech['ma50']>tech['ma200'] else "Death"} RSI={tech['rsi']:.1f} Stoch={tech['stoch_k']:.1f}/{tech['stoch_d']:.1f} MACD={tech['macd']:.4f}/{tech['signal']:.4f} VWAP=${tech['vwap']:.2f} Vol={tech['vol_ratio']:.1f}x BB=${tech['bb_lower']:.2f}-${tech['bb_upper']:.2f} S:{sup} R:{res}
{fib}
ATR(14)=${tech['atr']:.2f} ({tech['atr_pct']:.1f}%/day) | {fgs}
{obv_info}{rs_info}
BT: {bt}
RES: {research[:1500]}
MAC: {macro[:1000]}
EAR: {earnings[:1000]}
INS: {insider[:1000]}
FUND: {f}{portfolio_section}
{sec_section}
SOCIAL SENTIMENT (from X/Twitter via Grok):
{sentiment[:1000]}
1)SETUP 2)BACKTEST(<3=weak) 3)MACRO 4)TIMING 5)PLAN:Dir,Entry,Stop(>2xATR=${tech['atr']*2:.2f}),T1(1-4wk),T2(1-3mo),R:R,Size 6)KILLERS(3) 7)VOL:{tech['atr_pct']:.1f}%/day. Real prices."""

    r = claude_client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=3500,
        messages=[{"role": "user", "content": prompt}]
    )
    return r.content[0].text


def gemini_contrarian(tk, nm, tech, fd, research, trader, fg, etf, sec_context="", sentiment=""):
    """Stage 7: Contrarian Devil's Advocate (Gemini + Search)."""
    f = "\n".join([f"- {k}: {v}" for k, v in fd.items()])
    sec_prompt = ""
    if sec_context:
        sec_prompt = f"\nSEC FILING DATA (use this for deeper financial analysis — multi-year trends, balance sheet details, management commentary):\n{sec_context[:2500]}"
    return gcall(f"""CONTRARIAN. Reasons NOT to trade {nm} ({tk}). USE GOOGLE SEARCH.
${tech['price']:.2f} ATR={tech['atr_pct']:.1f}% RSI={tech['rsi']:.1f} {"Golden" if tech['ma50']>tech['ma200'] else "Death"}
FUND: {f}
RESEARCH: {research[:1500]}
TRADER: {trader[:2000]}
SOCIAL: {sentiment[:500]}
1)BEAR-downgrades,shorts,lawsuits 2)MISSED RISKS 3)TECH COUNTER 4)EARNINGS FLAGS 5)FATAL FLAW?(binary)->FATAL FLAW DETECTED 6)fail prob 0-100%""", ticker=tk, func_name="contrarian")


def claude_final_pm(tk, nm, tech, fd, research, macro, earnings, insider,
                    trader, contrarian, fg, bt, etf, rs=None, earn_prox=None,
                    portfolio_ctx="", cash_rec=None, risk_scenarios="", sec_context="", sentiment=""):
    """Stage 8: Portfolio Manager Final Decision (Claude Sonnet). Now portfolio-aware."""
    f = "\n".join([f"- {k}: {v}" for k, v in fd.items()])
    sec_prompt = ""
    if sec_context:
        sec_prompt = f"\nSEC FILING DATA (use this for deeper financial analysis — multi-year trends, balance sheet details, management commentary):\n{sec_context[:2500]}"
    fgl = (f"F&G:{fg['score']}/100({fg['label']})"
           if fg and fg["score"] else "F&G:N/A")

    # Earnings proximity warning
    earn_warning = ""
    if earn_prox and earn_prox[0] is not None:
        days = earn_prox[0]
        if days <= 14:
            earn_warning = f"\n⚠️ EARNINGS IN {days} DAYS ({earn_prox[1]}) — ELEVATED EVENT RISK. Consider waiting or reducing size."

    # Relative strength context
    rs_ctx = ""
    if rs and rs.get("verdict", "N/A") != "N/A":
        rs_ctx = f"\nSECTOR RS: {rs['verdict']}"

    # Portfolio + cash management context
    portfolio_section = ""
    if portfolio_ctx:
        portfolio_section = f"\n\n{portfolio_ctx}"

    cash_section = ""
    if cash_rec:
        rec_cash, regime, trims = cash_rec
        cash_section = f"\n\nCASH MANAGEMENT:\nRecommended cash: {rec_cash:.0f}% | Regime: {regime}"
        if trims:
            cash_section += "\nSuggested trims to raise cash:"
            for t in trims:
                cash_section += f"\n  {t['ticker']}: {t['current_pct']:.1f}% → {t['new_pct']:.1f}% (trim {t['trim_pct']:.1f}%)"

    risk_section = ""
    if risk_scenarios:
        risk_section = f"\n\nRISK SCENARIOS:\n{risk_scenarios}"

    # SEC filing narrative (MD&A + Risk Factors from 10-K/10-Q)
    sec_section = ""
    if sec_context:
        sec_section = f"\n\nSEC FILING CONTEXT:\n{sec_context[:4000]}"

    prompt = f"""FINAL PM. Steelman before reject. Bull=Bear->NO TRADE. FATAL FLAW->cap.
ATR=${tech['atr']:.2f}={tech['atr_pct']:.1f}%/day. SIZE:<2%:3-5%|2-3%:2-4%|3-4%:1-3%|>4%:max1-2%. FLAW:max1%. Stop>2xATR(>${tech['atr']*2:.2f}).
NOTE: SGOV, BIL, SHV, and similar T-bill/money market ETFs are treated as CASH equivalents in this portfolio. The cash % shown already includes these.
NOTE: Broad index funds (SPY, SPYM, VOO, VTI, QQQ, etc.) provide general equity exposure and should NOT be treated as single-sector concentration. SPYM is used as a long-term equity parking vehicle when cash is not in immediate use (1+ year timeline).
{nm}({tk}) ${tech['price']:.2f} RSI{tech['rsi']:.1f} {"Golden" if tech['ma50']>tech['ma200'] else "Death"} {fgl}{earn_warning}{rs_ctx}{portfolio_section}{cash_section}{risk_section}
FUND: {f}
BULL: Res:{research[:1200]}|Mac:{macro[:800]}|Earn:{earnings[:800]}|Ins:{insider[:800]}|Trader:{trader[:1500]}
SOCIAL SENTIMENT:
{sentiment[:1200]}
BEAR: {contrarian[:2000]}{sec_section}
BT: {bt}
1)SCOREBOARD 1-10
2)BULL vs BEAR
3)FATAL FLAW Y/N
4)ATR SIZE — factor portfolio context if provided. Consider sector concentration and correlation with existing holdings.
5)VERDICT,Entry,Stop,Targets,Size,Horizon
5b)SOCIAL vs FUNDAMENTAL DIVERGENCE: Does X sentiment agree with your analysis? If not, who is likely right?
6)PORTFOLIO FIT: How does this fit the current portfolio? Does it add diversification or concentrate risk?
7)CASH RECOMMENDATION: Given market regime, what should total cash allocation be? Should any positions be trimmed?
8)EXEC SUMMARY 5 lines."""

    r = claude_client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=4500,
        messages=[{"role": "user", "content": prompt}]
    )
    return r.content[0].text


# ============================================================================

In [11]:
# SECTION 7: HTML REPORT BUILDER
# ============================================================================



def get_financial_statements_html(ticker, etf=False):
    """
    Pull 3 years of Income Statement, Balance Sheet, and Cash Flow from yfinance
    and return a formatted HTML section for the report.

    Uses yfinance .financials, .balance_sheet, .cashflow DataFrames —
    separate from SEC XBRL data that feeds the AI prompts.
    Returns empty string for ETFs (no standard corporate financials).
    No API cost — yfinance is free and already in use.
    """
    if etf:
        return ""

    try:
        stk = yf.Ticker(ticker)

        def _fmt_val(val):
            """Format a raw dollar value into readable billions/millions."""
            if val is None or (isinstance(val, float) and (val != val)):  # NaN check
                return "\u2014"
            try:
                v = float(val)
                if abs(v) >= 1e9:
                    return f"${v/1e9:.2f}B"
                elif abs(v) >= 1e6:
                    return f"${v/1e6:.0f}M"
                elif abs(v) >= 1e3:
                    return f"${v/1e3:.0f}K"
                else:
                    return f"${v:.2f}"
            except (TypeError, ValueError):
                return "\u2014"

        def _build_table(df, rows_to_show, title, color):
            """
            Build one HTML table from a yfinance financial DataFrame.
            df columns are dates (most recent first), index is metric names.
            rows_to_show: list of label substrings to match (case-insensitive).
            """
            if df is None or df.empty:
                return ""

            # Limit to 3 most recent annual periods
            cols = df.columns[:3]
            years = [str(c.year) if hasattr(c, 'year') else str(c)[:4] for c in cols]

            header = "".join([
                f'<th style="text-align:right;padding:7px 14px;background:#f0f4f8;'
                f'font-size:11px;color:#555">{y}</th>'
                for y in years
            ])

            data_rows = ""
            row_count = 0
            for metric in rows_to_show:
                # Case-insensitive partial match — resilient across yfinance versions
                match = None
                for idx in df.index:
                    if metric.lower() in str(idx).lower():
                        match = idx
                        break
                if match is None:
                    continue

                bg = "background:#f9fafb;" if row_count % 2 == 0 else ""
                cells = ""
                for col in cols:
                    try:
                        raw = df.loc[match, col]
                        cells += (
                            f'<td style="text-align:right;padding:6px 14px;'
                            f'font-size:12px;font-weight:600;{bg}">{_fmt_val(raw)}</td>'
                        )
                    except Exception:
                        cells += '<td style="text-align:right;padding:6px 14px">\u2014</td>'

                data_rows += (
                    f'<tr>'
                    f'<td style="padding:6px 14px;font-size:12px;color:#444;{bg}">{match}</td>'
                    f'{cells}</tr>'
                )
                row_count += 1

            if row_count == 0:
                return ""

            return (
                f'<div style="margin-bottom:20px">'
                f'<h4 style="color:{color};font-size:13px;font-weight:700;margin-bottom:8px;'
                f'text-transform:uppercase;letter-spacing:0.5px">{title}</h4>'
                f'<div style="overflow-x:auto">'
                f'<table style="width:100%;border-collapse:collapse;font-family:\'Segoe UI\',sans-serif">'
                f'<thead><tr>'
                f'<th style="text-align:left;padding:7px 14px;background:#f0f4f8;'
                f'font-size:11px;color:#555">Metric</th>{header}'
                f'</tr></thead>'
                f'<tbody>{data_rows}</tbody>'
                f'</table></div></div>'
            )

        # Key metrics to display per statement
        income_metrics = [
            "Total Revenue", "Gross Profit", "Operating Income",
            "EBITDA", "Net Income", "Diluted EPS",
        ]
        balance_metrics = [
            "Total Assets", "Total Liabilities", "Total Stockholder Equity",
            "Cash And Cash Equivalents", "Total Debt", "Net Debt",
        ]
        cashflow_metrics = [
            "Operating Cash Flow", "Capital Expenditure",
            "Free Cash Flow", "Repurchase Of Capital Stock", "Dividends Paid",
        ]

        income_html  = _build_table(stk.financials,    income_metrics,   "Income Statement", "#1565c0")
        balance_html = _build_table(stk.balance_sheet, balance_metrics,  "Balance Sheet",    "#2e7d32")
        cash_html    = _build_table(stk.cashflow,      cashflow_metrics, "Cash Flow",        "#6a1b9a")

        content = income_html + balance_html + cash_html
        if not content.strip():
            return ""

        return (
            '<div class="section">'
            '<div class="st" style="color:#0d47a1;border-color:#0d47a1">'
            'Financial Statements \u2014 Last 3 Years</div>'
            '<div class="sb">'
            '<p style="font-size:11px;color:#999;margin-bottom:15px">'
            'Source: yfinance&nbsp;|&nbsp;Annual figures</p>'
            f'{content}'
            '</div></div>'
        )

    except Exception as e:
        log.warning(f"Financial statements table failed for {ticker}: {e}")
        return ""

def t2h(text):
    """Convert markdown-ish text to HTML paragraphs."""
    out = []
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            out.append('<br>')
        elif line.startswith('# '):
            out.append(f'<h3 style="color:#0d1b3e;margin:18px 0 8px">{line[2:]}</h3>')
        elif line.startswith('## '):
            out.append(f'<h4 style="color:#1a3a6e;margin:14px 0 6px">{line[3:]}</h4>')
        elif line.startswith(('- ', '* ')):
            inner = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", line[2:])
            out.append(f'<p style="margin:3px 0 3px 24px">&bull; {inner}</p>')
        else:
            inner = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", line)
            out.append(f'<p style="margin:5px 0;line-height:1.7">{inner}</p>')
    return '\n'.join(out)


def build_report(tk, nm, tech, fd, research, macro, earnings, insider,
                 trader, contrarian, pm, bt, fg, chart, etf, rs=None, earn_prox=None,
                 portfolio=None, cash_rec=None, risk_scenarios="", target_alloc=None, sec_data=None, sentiment=""):
    """Build the final HTML report. Now includes portfolio, risk, and cash sections."""
    d = tech

    # Color coding
    rsi_color = "#c62828" if d['rsi'] > 70 else "#2e7d32" if d['rsi'] < 30 else "#555"
    macd_color = "#2e7d32" if d['macd'] > d['signal'] else "#c62828"
    ma_color = "#2e7d32" if d['ma50'] > d['ma200'] else "#c62828"
    vwap_color = "#2e7d32" if d['price'] > d['vwap'] else "#c62828"
    atr_color = "#2e7d32" if d['atr_pct'] < 2 else "#e65100" if d['atr_pct'] < 4 else "#c62828"

    # Fear & Greed badge
    fg_html = ""
    if fg and fg["score"]:
        fgc = ("#c62828" if fg["score"] <= 25 else "#e65100" if fg["score"] <= 45
               else "#555" if fg["score"] <= 55 else "#2e7d32" if fg["score"] <= 75
               else "#1565c0")
        fg_html = (
            f'<div style="background:#f5f5f5;padding:15px;border-radius:10px;'
            f'margin:15px 0;text-align:center">'
            f'<span style="font-size:28px">{fg["emoji"]}</span>'
            f'<span style="font-size:22px;font-weight:700;color:{fgc};margin-left:10px">'
            f'{fg["score"]}/100</span>'
            f'<span style="font-size:14px;color:#666;margin-left:8px">{fg["label"]}</span>'
            f'</div>'
        )

    # NEW: Earnings proximity warning badge
    earn_badge = ""
    if earn_prox and earn_prox[0] is not None and earn_prox[0] <= 14:
        earn_badge = (
            f'<div style="background:#fff3e0;border:2px solid #e65100;padding:12px;'
            f'border-radius:10px;margin:10px 0;text-align:center;font-weight:700;'
            f'color:#e65100">⚠️ EARNINGS IN {earn_prox[0]} DAYS — {earn_prox[1]}</div>'
        )

    # Target allocation badge
    alloc_badge = ""
    if target_alloc is not None:
        alloc_badge = (
            f'<div style="background:#e3f2fd;border:2px solid #1565c0;padding:12px;'
            f'border-radius:10px;margin:10px 0;text-align:center;font-weight:700;'
            f'color:#1565c0">🎯 USER TARGET: {target_alloc:.1f}% allocation for {tk} — PM will evaluate below</div>'
        )

    # NEW: Relative Sector Strength badge
    rs_badge = ""
    if rs and rs.get("verdict", "N/A") != "N/A" and rs.get("sector_etf", "N/A") != "N/A":
        rs_badge = (
            f'<div style="background:#f5f5f5;padding:10px;border-radius:8px;'
            f'margin:8px 0;text-align:center;font-size:13px">'
            f'<strong>Sector RS vs {rs["sector_etf"]}:</strong> '
            f'20d {rs.get("rs_20d","N/A")} | 60d {rs.get("rs_60d","N/A")} | '
            f'120d {rs.get("rs_120d","N/A")} — {rs["verdict"]}</div>'
        )

    # SEC Filing badge
    sec_badge = ""
    if sec_data and sec_data.get("source", "N/A") != "N/A":
        sec_badge = (
            f'<div style="background:#f3e5f5;padding:10px;border-radius:8px;'
            f'margin:8px 0;text-align:center;font-size:13px">'
            f'<strong>SEC Filing Data:</strong> {sec_data["source"]} '
            f'(filed {sec_data.get("filing_date", "N/A")}) — '
            f'Multi-year financials + MD&A narrative included in AI analysis</div>'
        )

    # Financial statements tables (Income / Balance Sheet / Cash Flow)
    # Built from yfinance DataFrames; no extra API cost
    fin_statements_html = get_financial_statements_html(tk, etf)

    # Fundamentals data grid
    fund_rows = ""
    items = list(fd.items())
    for i in range(0, len(items), 3):
        for j in range(3):
            if i + j < len(items):
                k, v = items[i + j]
                bg = "background:#f8f9fb;" if (i // 3) % 2 == 1 else ""
                fund_rows += (
                    f'<div class="dc" style="{bg}">'
                    f'<div class="dl">{k}</div><div class="dv">{v}</div></div>'
                )

    # Portfolio context section
    portfolio_html = ""
    if portfolio and portfolio.get("holdings"):
        pf_rows = ""
        for ticker, pct in portfolio["holdings"].items():
            sec = portfolio.get("sectors", {}).get(ticker, "?")
            beta = portfolio.get("betas", {}).get(ticker)
            beta_str = f"β={beta:.2f}" if beta else ""
            pf_rows += (
                f'<div class="dc"><div class="dl">{ticker}</div>'
                f'<div class="dv">{pct:.1f}% <span style="font-size:10px;color:#888">{sec} {beta_str}</span></div></div>'
            )
        pf_rows += f'<div class="dc" style="background:#e8f5e9"><div class="dl">CASH</div><div class="dv">{portfolio["cash_pct"]:.1f}%</div></div>'

        if portfolio.get("portfolio_beta"):
            pf_rows += f'<div class="dc" style="background:#fff3e0"><div class="dl">Portfolio Beta</div><div class="dv">{portfolio["portfolio_beta"]:.2f}</div></div>'

        # Sector concentration
        sector_html = ""
        if portfolio.get("sector_weights"):
            sector_items = sorted(portfolio["sector_weights"].items(), key=lambda x: -x[1])
            for sec, wt in sector_items:
                color = "#c62828" if wt > 30 else "#e65100" if wt > 20 else "#333"
                sector_html += f'<span style="margin-right:12px;color:{color};font-weight:600">{sec}: {wt:.0f}%</span>'

        # Correlation with new ticker
        corr_html = ""
        if portfolio.get("correlations_with_new"):
            high_corrs = {k: v for k, v in portfolio["correlations_with_new"].items() if abs(v) > 0.6}
            if high_corrs:
                corr_items = []
                for pair, corr in sorted(high_corrs.items(), key=lambda x: -abs(x[1])):
                    color = "#c62828" if abs(corr) > 0.8 else "#e65100"
                    corr_items.append(f'<span style="color:{color};font-weight:600">{pair}: {corr:.2f}</span>')
                corr_html = f'<div style="margin-top:8px;font-size:12px">Correlation with {tk}: {" | ".join(corr_items)}</div>'

        portfolio_html = f"""
        <div class="section">
          <div class="st" style="color:#1565c0;border-color:#1565c0">Current Portfolio</div>
          <div class="sb">
            <div class="dg">{pf_rows}</div>
            <div style="margin-top:10px;font-size:12px">{sector_html}</div>
            {corr_html}
          </div>
        </div>"""

    # Cash recommendation section
    cash_html = ""
    if cash_rec:
        rec_cash, regime, trims = cash_rec
        current_cash = portfolio["cash_pct"] if portfolio else 0
        cash_color = "#2e7d32" if abs(rec_cash - current_cash) < 5 else "#e65100"

        trim_html = ""
        if trims:
            trim_items = "".join([
                f'<div style="padding:4px 0;font-size:13px">'
                f'<strong>{t["ticker"]}</strong>: {t["current_pct"]:.1f}% → {t["new_pct"]:.1f}% '
                f'<span style="color:#c62828">(trim {t["trim_pct"]:.1f}%)</span></div>'
                for t in trims
            ])
            trim_html = f'<div style="margin-top:10px;padding:10px;background:#fff3e0;border-radius:6px"><strong>Suggested Trims:</strong>{trim_items}</div>'

        cash_html = f"""
        <div class="section">
          <div class="st" style="color:#00695c;border-color:#00695c">Cash Management</div>
          <div class="sb">
            <div style="display:flex;gap:20px;align-items:center;flex-wrap:wrap">
              <div><span style="font-size:10px;color:#888;text-transform:uppercase">Current Cash</span><br>
                <span style="font-size:22px;font-weight:700">{current_cash:.0f}%</span></div>
              <div style="font-size:24px;color:#888">→</div>
              <div><span style="font-size:10px;color:#888;text-transform:uppercase">Recommended</span><br>
                <span style="font-size:22px;font-weight:700;color:{cash_color}">{rec_cash:.0f}%</span></div>
            </div>
            <div style="margin-top:8px;font-size:13px;color:#555">{regime}</div>
            {trim_html}
          </div>
        </div>"""

    # Risk scenarios section
    # SEC Filing narrative section (MD&A + Risk Factors from actual filings)
    sec_filing_html = ""
    if sec_data and (sec_data.get("mda_text") or sec_data.get("risk_factors_text")):
        sec_parts = []
        if sec_data.get("mda_text"):
            # Clean up for HTML display
            mda_display = sec_data["mda_text"][:3000].replace("\n", "<br>")
            sec_parts.append(f'<h4 style="color:#6a1b9a;margin:10px 0 5px">Management Discussion & Analysis</h4><p style="font-size:13px;line-height:1.6">{mda_display}</p>')
        if sec_data.get("risk_factors_text"):
            risk_display = sec_data["risk_factors_text"][:2000].replace("\n", "<br>")
            sec_parts.append(f'<h4 style="color:#b71c1c;margin:10px 0 5px">Key Risk Factors</h4><p style="font-size:13px;line-height:1.6">{risk_display}</p>')
        sec_filing_html = f"""
        <div class="section">
          <div class="st" style="color:#6a1b9a;border-color:#6a1b9a">SEC Filing — {sec_data.get('source', 'N/A')} (Filed {sec_data.get('filing_date', 'N/A')})</div>
          <div class="sb">{''.join(sec_parts)}</div>
        </div>"""

    risk_html = ""
    if risk_scenarios:
        risk_html = f"""
        <div class="section">
          <div class="st" style="color:#880e4f;border-color:#880e4f">Risk Scenarios</div>
          <div class="sb" style="font-family:monospace;font-size:12px;white-space:pre-wrap;background:#f9f9f9;padding:15px">{risk_scenarios}</div>
        </div>"""

    def section(title, css_class, body):
        return (
            f'<div class="section"><div class="st {css_class}">{title}</div>'
            f'<div class="sb">{t2h(body)}</div></div>'
        )

    css = """
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:-apple-system,BlinkMacSystemFont,'Inter','Segoe UI',Roboto,Oxygen,Ubuntu,Cantarell,sans-serif;background:#f4f7f9;color:#2c3e50}
    .ctr{max-width:1100px;margin:40px auto;background:#fff;box-shadow:0 12px 36px rgba(0,0,0,0.08);border-radius:16px;overflow:hidden}
    .hdr{background:linear-gradient(135deg,#1e3c72,#2a5298);color:#fff;padding:36px 45px}
    .hdr h1{font-size:30px;font-weight:800;letter-spacing:-0.5px;margin-bottom:6px}
    .hdr .s{opacity:.85;font-size:14px;font-weight:500;letter-spacing:0.5px}
    .section{background:#fff;padding:35px 45px;border-bottom:1px solid #edf2f7}
    .st{font-size:19px;font-weight:700;padding-bottom:12px;margin-bottom:20px;border-bottom:3px solid;letter-spacing:-0.2px}
    .sb{font-size:15px;line-height:1.75;color:#34495e}
    .res{color:#2980b9;border-color:#2980b9}
    .mac{color:#8e44ad;border-color:#8e44ad}
    .ear{color:#16a085;border-color:#16a085}
    .ins{color:#d35400;border-color:#d35400}
    .bkt{color:#27ae60;border-color:#27ae60}
    .trd{color:#009432;border-color:#009432}
    .con{color:#c0392b;border-color:#c0392b}
    .pm{color:#e67e22;border-color:#e67e22}
    .snap{background:#fcfcfd;padding:30px 45px;border-bottom:1px solid #edf2f7}
    .pr{display:flex;align-items:center;gap:15px;margin-top:15px;flex-wrap:wrap}
    .price{font-size:44px;font-weight:800;color:#1e3c72;letter-spacing:-1.5px}
    .badge{font-size:12px;font-weight:700;padding:6px 14px;border-radius:8px;text-transform:uppercase;letter-spacing:0.5px;box-shadow:0 2px 4px rgba(0,0,0,0.05)}
    .dg{display:grid;grid-template-columns:repeat(3,1fr);margin-top:25px;border:1px solid #edf2f7;border-radius:12px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.02)}
    .dc{padding:14px 20px;border-bottom:1px solid #edf2f7;border-right:1px solid #edf2f7;background:#fff}
    .dl{font-size:11px;color:#7f8c8d;text-transform:uppercase;font-weight:700;letter-spacing:0.5px}
    .dv{font-size:14px;font-weight:700;color:#2c3e50;margin-top:6px}
    .cht{background:#fff;padding:35px 45px;border-bottom:1px solid #edf2f7;text-align:center}
    .cht img{max-width:100%;border-radius:12px;box-shadow:0 4px 16px rgba(0,0,0,0.06)}
    .pms{background:linear-gradient(to bottom,#fffaf0,#fff)}
    .cons{background:linear-gradient(to bottom,#fdf2f2,#fff)}
    .ftr{text-align:center;color:#95a5a6;font-size:12px;padding:25px;background:#f8f9fa}
    .pip{display:flex;justify-content:center;align-items:center;gap:8px;margin:20px 0 30px;flex-wrap:wrap}
    .pip span{background:#edf2f7;color:#4a5568;padding:8px 16px;border-radius:24px;font-size:11px;font-weight:700;letter-spacing:0.5px;text-transform:uppercase;box-shadow:inset 0 1px 2px rgba(255,255,255,0.5)}
    .pip .final{background:#fffaf0;border:1px solid #e67e22;color:#e67e22;box-shadow:0 2px 6px rgba(230,126,34,0.15)}
    .pip .devil{background:#fdf2f2;border:1px solid #c0392b;color:#c0392b;box-shadow:0 2px 6px rgba(192,57,43,0.15)}
    .pb{position:fixed;bottom:30px;right:30px;background:#1e3c72;color:#fff;border:none;padding:14px 28px;border-radius:30px;font-weight:600;font-size:15px;cursor:pointer;z-index:1000;box-shadow:0 6px 20px rgba(30,60,114,0.4);transition:transform 0.2s, box-shadow 0.2s}
    .pb:hover{transform:translateY(-2px);box-shadow:0 8px 25px rgba(30,60,114,0.5)}
    @media print{.pb,.ctr{box-shadow:none;margin:0}.hdr{border-radius:0}}
    """

    return f"""<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>{tk} Analysis</title>
<style>{css}</style></head><body>
<button class="pb" onclick="window.print()">Print</button>
<div class="ctr">
  <div class="hdr">
    <h1>AI Stock Analyzer</h1>
    <div class="s">{nm} ({tk}) &mdash; {datetime.now().strftime('%B %d, %Y')}</div>
  </div>
  <div class="snap">
    <div class="pip">
      <span>Data+ATR</span><span>&#8594;</span><span>Research</span><span>&#8594;</span>
      <span>Macro</span><span>&#8594;</span><span>Earnings</span><span>&#8594;</span>
      <span>Insider</span><span>&#8594;</span><span>Trader</span><span>&#8594;</span>
      <span class="devil">Contrarian</span><span>&#8594;</span><span class="final">Final PM</span>
    </div>
    <div class="pr">
      <span class="price">${d['price']:.2f}</span>
      <span class="badge" style="background:{ma_color}15;color:{ma_color}">{"Golden" if d['ma50']>d['ma200'] else "Death"}</span>
      <span class="badge" style="background:{rsi_color}15;color:{rsi_color}">RSI {d['rsi']:.0f}</span>
      <span class="badge" style="background:{macd_color}15;color:{macd_color}">MACD {"Bull" if d['macd']>d['signal'] else "Bear"}</span>
      <span class="badge" style="background:{atr_color}15;color:{atr_color}">ATR {d['atr_pct']:.1f}%</span>
    </div>
    {earn_badge}
    {alloc_badge}
    {fg_html}
    {rs_badge}
    {sec_badge}
    <div class="dg">{fund_rows}</div>
  </div>
  <div class="cht"><img src="data:image/png;base64,{chart}"></div>
  {fin_statements_html}
  {portfolio_html}
  {cash_html}
  {risk_html}
  {sec_filing_html}
  <div class="section"><div class="st bkt">Backtest 2yr</div>
    <div class="sb" style="font-family:monospace;font-size:12px;white-space:pre-wrap;background:#f9f9f9;padding:15px">{bt}</div>
  </div>
  {section("Equity Researcher — Gemini", "res", research)}
  {section("Macro Analyst — Gemini", "mac", macro)}
  {section("Earnings Scout — Gemini", "ear", earnings)}
  {section("Insider Flow — Gemini", "ins", insider)}
  <div class="section">
    <div class="st" style="color:#1DA1F2;border-color:#1DA1F2">Social Sentiment (X/Twitter) — Grok 3 Mini Fast</div>
    <div class="sb">{t2h(sentiment)}</div>
  </div>
  {section("Trader — Claude Sonnet 4.5", "trd", trader)}
  <div class="section cons">
    <div class="st con">Contrarian — Gemini (Fatal Flaw Check)</div>
    <div class="sb">{t2h(contrarian)}</div>
  </div>
  <div class="section pms">
    <div class="st pm">Final PM — Claude Sonnet 4.5 (Portfolio-Aware)</div>
    <div class="sb">{t2h(pm)}</div>
  </div>
  <div class="ftr">AI Stock Analyzer — Educational only. Not financial advice.</div>
</div></body></html>"""


# ============================================================================

In [ ]:
# SECTION 8: MAIN PIPELINE + WATCHLIST
# ============================================================================

def run_full(ti, portfolio_text="", target_alloc_text="", progress=gr.Progress()):
    """Execute the full 8-stage analysis pipeline. Now portfolio-aware."""
    tk = ti.strip().upper()
    if not tk:
        return "Enter a ticker.", ""

    try:
        # --- Parse portfolio if provided ---
        portfolio = parse_portfolio(portfolio_text)
        if portfolio:
            log.info(f"Portfolio parsed: {len(portfolio['holdings'])} holdings, {portfolio['cash_pct']:.1f}% cash")

        # --- Parse target allocation ---
        target_alloc = None
        if target_alloc_text and target_alloc_text.strip():
            try:
                target_alloc = float(target_alloc_text.strip().replace('%', ''))
                log.info(f"User target allocation: {target_alloc:.1f}%")
            except ValueError:
                log.warning(f"Could not parse target allocation: {target_alloc_text}")

        # --- Stage 1: Data + ATR ---
        progress(0.03, "Stage 1/8: Data + ATR...")
        df, info, err = get_stock_data(tk)
        if err:
            return f"Error: {err}", ""

        etf = is_etf(info)
        nm = info.get("longName", tk)
        sector = info.get("sector", info.get("category", "Unknown"))

        progress(0.07, "Calculating indicators...")
        df = calculate_indicators(df)
        latest = df.iloc[-1]

        sups, ress = get_support_resistance(df)
        fibs, fh, fl = get_fibonacci_levels(df)
        atr_val = latest["ATR"]
        atr_pct = (atr_val / latest["Close"]) * 100

        # OBV trend detection
        obv_bullish = latest["OBV"] > latest["OBV_MA20"]

        tech = {
            "price": latest["Close"],
            "ma20": latest["MA20"],
            "ma50": latest["MA50"],
            "ma200": latest["MA200"],
            "rsi": latest["RSI"],
            "macd": latest["MACD"],
            "signal": latest["Signal"],
            "bb_upper": latest["BB_Upper"],
            "bb_lower": latest["BB_Lower"],
            "bb_mid": latest["BB_Mid"],
            "vol_ratio": (latest["Volume"] / latest["Vol_MA20"]
                          if latest["Vol_MA20"] > 0 else 1),
            "stoch_k": latest["Stoch_K"],
            "stoch_d": latest["Stoch_D"],
            "vwap": latest["VWAP"],
            "supports": sups,
            "resistances": ress,
            "fib_levels": fibs,
            "fib_high": fh,
            "fib_low": fl,
            "atr": atr_val,
            "atr_pct": atr_pct,
            "obv_bullish": obv_bullish,
        }

        # fd is now set below after SEC data is fetched (get_enhanced_fundamentals)

        # --- Parallel: F&G, Earnings Proximity, Sector RS, SEC Filing Data ---
        progress(0.10, "Fear & Greed + Sector + SEC Filings...")
        with ThreadPoolExecutor(max_workers=4) as ex:
            fut_fg   = ex.submit(get_fear_greed)
            fut_earn = ex.submit(get_earnings_proximity, tk) if not etf else None
            fut_rs   = ex.submit(get_relative_sector_strength, tk, info, df) if not etf else None
            fut_sec  = ex.submit(get_sec_filing_data, tk, etf) if not etf else None
        fg        = fut_fg.result()
        earn_prox = fut_earn.result() if fut_earn else (None, "N/A")
        rs        = fut_rs.result() if fut_rs else None
        sec_data  = fut_sec.result() if fut_sec else None

        # Build enhanced fundamentals (yfinance snapshot + SEC deep data)
        fd, sec_context = "", ""  # safe defaults in case get_enhanced_fundamentals raises
        fd, sec_context = get_enhanced_fundamentals(info, sec_data, etf)

        # --- Portfolio enrichment ---
        portfolio_ctx = ""
        cash_rec = None
        risk_scenarios_text = ""
        if portfolio:
            progress(0.12, "Enriching portfolio data...")
            portfolio = enrich_portfolio(portfolio)
            portfolio = compute_new_ticker_correlations(portfolio, tk)
            portfolio_ctx = format_portfolio_context(portfolio, new_ticker=tk)

            # Append user's target allocation to context
            if target_alloc is not None:
                portfolio_ctx += f"\n\nUSER'S PROPOSED ALLOCATION: {target_alloc:.1f}% for {tk}"
                portfolio_ctx += f"\nThe PM MUST evaluate whether {target_alloc:.1f}% is appropriate given ATR, portfolio concentration, and market regime. Agree, adjust, or reject with reasoning."

            # Cash recommendation
            cash_rec = recommend_cash_position(fg, portfolio, tech)

            # Risk scenarios — use target_alloc if provided, else ATR-based default
            scenario_alloc = target_alloc if target_alloc is not None else 3.0
            if target_alloc is None:
                if tech["atr_pct"] < 2:
                    scenario_alloc = 4.0
                elif tech["atr_pct"] > 4:
                    scenario_alloc = 1.5
            risk_scenarios_text = run_risk_scenarios(tech, portfolio, scenario_alloc, info)
        elif target_alloc is not None:
            # No portfolio but user gave a target — still run risk scenarios without portfolio context
            portfolio_ctx = f"USER'S PROPOSED ALLOCATION: {target_alloc:.1f}% for {tk}\nNo current portfolio provided. The PM should evaluate whether {target_alloc:.1f}% is appropriate given ATR and market conditions."
            risk_scenarios_text = run_risk_scenarios(tech, None, target_alloc, info)

        progress(0.13, "Chart...")
        chart = chart_to_b64(plot_charts(df, tk))

        progress(0.16, "Backtesting...")
        bt = run_backtest(df, tech)

        # --- Stages 2-5: 4x Gemini in parallel ---
        progress(0.22, "Stages 2-5b: Gemini + Grok parallel...")
        with ThreadPoolExecutor(max_workers=5) as ex:
            fut_research  = ex.submit(gemini_research, tk, nm, fd, etf, sec_context)
            fut_macro     = ex.submit(gemini_macro, tk, nm, sector, etf)
            fut_earnings  = ex.submit(gemini_earnings, tk, nm, etf)
            fut_insider   = ex.submit(gemini_insider, tk, nm, etf)
            fut_sentiment = ex.submit(grok_social_sentiment, tk, nm, etf)  # Stage 5b: X/Twitter via Grok

            research  = fut_research.result()
            macro     = fut_macro.result()
            earnings  = fut_earnings.result()
            insider   = fut_insider.result()
            sentiment = fut_sentiment.result()

        # --- Stage 6: Trader ---
        progress(0.50, "Stage 6/8: Trader (Claude)...")
        trader = claude_trader(tk, nm, tech, fd, research, macro, earnings,
                               insider, fg, bt, etf, rs=rs, portfolio_ctx=portfolio_ctx,
                               sec_context=sec_context, sentiment=sentiment)

        # --- Stage 7: Contrarian ---
        progress(0.65, "Stage 7/8: Contrarian (Gemini)...")
        contrarian = gemini_contrarian(tk, nm, tech, fd, research, trader, fg, etf, sec_context=sec_context, sentiment=sentiment)

        # --- Stage 8: Final PM ---
        progress(0.80, "Stage 8/8: Final PM (Claude)...")
        pm = claude_final_pm(tk, nm, tech, fd, research, macro, earnings, insider,
                             trader, contrarian, fg, bt, etf, rs=rs, earn_prox=earn_prox,
                             portfolio_ctx=portfolio_ctx, cash_rec=cash_rec,
                             risk_scenarios=risk_scenarios_text, sec_context=sec_context, sentiment=sentiment)

        # --- Build Report ---
        progress(0.95, "Building report...")
        html = build_report(tk, nm, tech, fd, research, macro, earnings, insider,
                            trader, contrarian, pm, bt, fg, chart, etf,
                            rs=rs, earn_prox=earn_prox, portfolio=portfolio,
                            cash_rec=cash_rec, risk_scenarios=risk_scenarios_text,
                            target_alloc=target_alloc, sec_data=sec_data, sentiment=sentiment)

        b64 = base64.b64encode(html.encode()).decode()
        progress(1.0, "Done!")

        # Earnings warning in status
        earn_status = ""
        if earn_prox and earn_prox[0] is not None and earn_prox[0] <= 14:
            earn_status = f" | ⚠️ Earnings in {earn_prox[0]}d"

        sec_status = ""
        if sec_data and sec_data.get("source", "N/A") != "N/A":
            sec_status = f" | SEC:{sec_data['source']}"
        status = f"**{nm} ({tk})** done (ATR:{atr_pct:.1f}%{earn_status}{sec_status})"

        output_html = f"""<div style="text-align:center;padding:20px">
<h2 style="color:#1a3a6e">{nm} ({tk})</h2>
<p>8 stages | ATR:{atr_pct:.1f}%{earn_status}</p>
<button onclick="var w=window.open();w.document.write(decodeURIComponent(escape(atob('{b64}'))));w.document.close();"
  style="background:#1a3a6e;color:#fff;border:none;padding:12px 24px;border-radius:8px;font-size:15px;cursor:pointer;margin:8px">
  Open Report</button><br>
<a href="data:text/html;base64,{b64}" download="{tk}_{datetime.now().strftime('%Y%m%d')}.html"
  style="color:#1565c0">Download HTML</a>
</div>
<script>var w=window.open();if(w){{w.document.write(decodeURIComponent(escape(atob('{b64}'))));w.document.close();}}</script>"""

        return status, output_html

    except Exception as e:
        log.error(f"Pipeline failed for {tk}: {e}", exc_info=True)
        return f"Error analyzing {tk}: {str(e)}", ""


def screen_tickers(tickers):
    """
    Run technical scorecard on a list of tickers.
    Returns (rows_text, close_data_dict) for correlation analysis.
    """
    header = f"{'Tick':<7} {'Price':<9} {'RSI':<6} {'ATR%':<6} {'Sharpe':<7} {'MA':<10} {'MACD':<7} Verdict"
    rows = [header, "-" * 75]
    close_data = {}

    for t in tickers:
        try:
            df, info, err = get_stock_data(t)
            if err:
                rows.append(f"{t:<7} ERROR: {err}")
                continue

            df = calculate_indicators(df)
            lat = df.iloc[-1]
            atr_pct = (lat["ATR"] / lat["Close"]) * 100

            if len(df) >= 60:
                close_data[t] = df["Close"].tail(60).pct_change().dropna().values

            if len(df) >= 60:
                rets = df["Close"].tail(60).pct_change().dropna()
                sharpe = (rets.mean() / (rets.std() + 1e-9)) * np.sqrt(252)
                sharpe_str = f"{sharpe:.2f}"
            else:
                sharpe_str = "N/A"

            score = sum([
                1 if lat["MA50"] > lat["MA200"] else 0,
                1 if 40 < lat["RSI"] < 65 else 0,
                1 if lat["MACD"] > lat["Signal"] else 0,
            ])

            ma_label = "Gold" if lat["MA50"] > lat["MA200"] else "Death"
            macd_label = "Bull" if lat["MACD"] > lat["Signal"] else "Bear"
            verdict = "BUY" if score == 3 else "HOLD" if score == 2 else "AVOID"

            rows.append(
                f"{t:<7} ${lat['Close']:<8.2f} {lat['RSI']:<6.1f} {atr_pct:<5.1f}% "
                f"{sharpe_str:<7} {ma_label:<10} {macd_label:<7} {verdict}"
            )

        except Exception as e:
            rows.append(f"{t:<7} ERROR: {str(e)[:30]}")
            log.warning(f"Scorecard scan failed for {t}: {e}")

    return rows, close_data


def correlation_warnings(close_data):
    """Generate correlation warnings for a set of tickers."""
    rows = []
    if len(close_data) >= 2:
        rows.append("")
        rows.append("CORRELATION WARNINGS (>0.80):")
        has_warning = False
        tklist = list(close_data.keys())
        for i in range(len(tklist)):
            for j in range(i + 1, len(tklist)):
                t1, t2 = tklist[i], tklist[j]
                try:
                    min_len = min(len(close_data[t1]), len(close_data[t2]))
                    corr = np.corrcoef(
                        close_data[t1][:min_len],
                        close_data[t2][:min_len]
                    )[0, 1]
                    if abs(corr) > 0.80:
                        rows.append(f"  ⚠️ {t1} & {t2}: {corr:.2f} — high correlation, limited diversification")
                        has_warning = True
                except Exception:
                    pass
        if not has_warning:
            rows.append("  ✅ No high correlations detected — good diversification.")
    return rows


def get_ai_recommendations(portfolio, fg):
    """
    Use Claude to suggest 5-8 tickers that complement the current portfolio.
    Considers sector gaps, market regime, and diversification.
    """
    if portfolio is None:
        return []

    holdings_str = ", ".join([f"{t} ({p:.0f}%)" for t, p in portfolio["holdings"].items()])
    cash_str = f"{portfolio['cash_pct']:.0f}%"

    sector_str = "None identified"
    if portfolio.get("sector_weights"):
        sector_str = ", ".join([f"{s}: {w:.0f}%" for s, w in
                                sorted(portfolio["sector_weights"].items(), key=lambda x: -x[1])])

    high_corr_str = "None"
    if portfolio.get("correlations"):
        high = {k: v for k, v in portfolio["correlations"].items() if abs(v) > 0.7}
        if high:
            high_corr_str = ", ".join([f"{k}={v:.2f}" for k, v in high.items()])

    beta_str = f"{portfolio['portfolio_beta']:.2f}" if portfolio.get("portfolio_beta") else "Unknown"

    fg_str = f"{fg['score']}/100 ({fg['label']})" if fg and fg.get("score") else "Unknown"

    # Identify index funds in holdings
    index_funds = portfolio.get("index_funds", {})
    index_str = ""
    if index_funds:
        idx_list = ", ".join([f"{t} ({p:.0f}%)" for t, p in index_funds.items()])
        index_str = f"\nIndex fund holdings (general equity exposure, NOT single-sector bets): {idx_list}"

    prompt = f"""You are a portfolio strategist. Analyze this portfolio and recommend 5-8 specific stock tickers that would COMPLEMENT it well.

CURRENT PORTFOLIO:
Holdings: {holdings_str}
Cash: {cash_str}
Sector exposure (adjusted): {sector_str}
High correlations: {high_corr_str}
Portfolio beta: {beta_str}
Market sentiment (Fear & Greed): {fg_str}{index_str}

IMPORTANT CONTEXT:
- Broad index funds like SPY, SPYM, VOO, QQQ provide diversified equity exposure across many sectors. Do NOT treat them as single-sector concentration.
- SPYM/SPY-type holdings are used as long-term equity parking vehicles (1+ year timeline) when cash is not in immediate use.
- SGOV and similar T-bill ETFs are already counted as cash above.

RULES:
- Do NOT recommend tickers already in the portfolio
- Do NOT recommend broad index funds or ETFs — suggest individual stocks or sector-specific ETFs
- Focus on filling SECTOR GAPS (underweight sectors)
- Consider the current market regime from Fear & Greed
- Mix of growth and value based on market conditions
- Include at least 1 defensive/low-correlation pick
- Include at least 1 high-conviction growth pick
- If portfolio is high-beta, suggest some low-beta names to balance
- If portfolio is concentrated in one sector, prioritize OTHER sectors

RESPOND WITH ONLY a JSON array of objects, no other text:
[
  {{"ticker": "XYZ", "reason": "one sentence why this fits"}},
  ...
]"""

    try:
        r = claude_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )
        text = r.content[0].text.strip()

        # Parse JSON — handle markdown fences
        text = text.replace("```json", "").replace("```", "").strip()
        recs = json.loads(text)

        # Filter out any tickers already in portfolio
        existing = set(portfolio["holdings"].keys())
        recs = [r for r in recs if r.get("ticker", "").upper() not in existing]

        return recs[:8]

    except Exception as e:
        log.error(f"AI recommendation failed: {e}")
        return []


def run_portfolio_review(portfolio_text, progress=gr.Progress()):
    """
    Portfolio review: scorecard on holdings, sector analysis,
    AI-recommended complementary tickers, and screening.
    """
    portfolio = parse_portfolio(portfolio_text)
    if portfolio is None or not portfolio["holdings"]:
        return "Enter your portfolio. Format: AAPL:15%, NVDA:10%, CASH:20%"

    output = []
    output.append("=" * 75)
    output.append("PORTFOLIO REVIEW")
    output.append("=" * 75)

    # --- Enrich portfolio ---
    progress(0.05, "Analyzing portfolio...")
    portfolio = enrich_portfolio(portfolio)

    # --- Scorecard on existing holdings ---
    progress(0.15, "Screening current holdings...")
    tickers = list(portfolio["holdings"].keys())
    output.append(f"\n📊 CURRENT HOLDINGS ({len(tickers)} positions + {portfolio['cash_pct']:.0f}% cash)")
    output.append("")

    rows, close_data = screen_tickers(tickers)
    output.extend(rows)

    # --- Correlation warnings ---
    output.extend(correlation_warnings(close_data))

    # --- Sector analysis ---
    output.append("")
    output.append("📁 SECTOR EXPOSURE:")
    if portfolio.get("sector_weights"):
        for sec, wt in sorted(portfolio["sector_weights"].items(), key=lambda x: -x[1]):
            bar = "█" * int(wt / 2)
            flag = " ⚠️ CONCENTRATED" if wt > 25 else ""
            output.append(f"  {sec:<35} {wt:5.1f}% {bar}{flag}")

    # Missing major sectors
    all_sectors = {"Technology", "Healthcare", "Financial Services", "Energy",
                   "Consumer Cyclical", "Consumer Defensive", "Industrials",
                   "Real Estate", "Utilities", "Communication Services", "Basic Materials"}
    held_sectors = set()
    for sec in portfolio.get("sector_weights", {}).keys():
        for s in all_sectors:
            if s.lower() in sec.lower():
                held_sectors.add(s)
    missing = all_sectors - held_sectors
    if missing:
        output.append(f"\n  Underweight / Missing: {', '.join(sorted(missing))}")

    # --- Portfolio beta ---
    if portfolio.get("portfolio_beta"):
        output.append(f"\n📈 Portfolio Beta: {portfolio['portfolio_beta']:.2f}")

    # --- Cash recommendation ---
    progress(0.30, "Market regime analysis...")
    fg = get_fear_greed()
    cash_rec = recommend_cash_position(fg, portfolio, {"atr_pct": 2.0})
    rec_cash, regime, trims = cash_rec

    output.append("")
    output.append(f"💰 CASH MANAGEMENT:")
    output.append(f"  Current cash: {portfolio['cash_pct']:.0f}%")
    output.append(f"  Recommended:  {rec_cash:.0f}%")
    output.append(f"  Regime: {regime}")

    if trims:
        output.append("  Suggested trims:")
        for t in trims:
            output.append(f"    {t['ticker']}: {t['current_pct']:.1f}% → {t['new_pct']:.1f}% (trim {t['trim_pct']:.1f}%)")

    # --- AI Recommendations ---
    progress(0.45, "AI generating recommendations...")
    output.append("")
    output.append("=" * 75)
    output.append("🤖 AI-RECOMMENDED ADDITIONS (Claude Sonnet 4.5)")
    output.append("=" * 75)

    recs = get_ai_recommendations(portfolio, fg)

    if recs:
        for i, rec in enumerate(recs, 1):
            ticker = rec.get("ticker", "?").upper()
            reason = rec.get("reason", "")
            output.append(f"  {i}. {ticker} — {reason}")

        # Screen the recommended tickers
        rec_tickers = [r["ticker"].upper() for r in recs if r.get("ticker")]
        if rec_tickers:
            progress(0.60, f"Screening {len(rec_tickers)} recommendations...")
            output.append("")
            output.append("📊 RECOMMENDATION SCORECARD:")
            output.append("")
            rec_rows, rec_close = screen_tickers(rec_tickers)
            output.extend(rec_rows)

            # Correlation of recs with existing portfolio
            all_close = {**close_data, **rec_close}
            output.append("")
            output.append("CORRELATION: RECOMMENDATIONS vs HOLDINGS:")
            for rec_t in rec_tickers:
                if rec_t not in all_close:
                    continue
                for hold_t in tickers:
                    if hold_t not in all_close:
                        continue
                    try:
                        min_len = min(len(all_close[rec_t]), len(all_close[hold_t]))
                        if min_len > 10:
                            corr = np.corrcoef(
                                all_close[rec_t][:min_len],
                                all_close[hold_t][:min_len]
                            )[0, 1]
                            if abs(corr) > 0.7:
                                flag = " ⚠️ HIGH" if abs(corr) > 0.8 else ""
                                output.append(f"  {rec_t} ↔ {hold_t}: {corr:.2f}{flag}")
                    except Exception:
                        pass
    else:
        output.append("  Could not generate recommendations. Try again.")

    progress(1.0, "Done!")
    output.append("")
    output.append("─" * 75)
    output.append("Use the Full Analysis tab to deep-dive any ticker above.")
    output.append("Educational purposes only. Not financial advice.")

    return "\n".join(output)


# ============================================================================
# SECTION 9: GRADIO UI
# ============================================================================

with gr.Blocks(title="AI Stock Analyzer", theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate")) as app:
    gr.Markdown(
        "# 📈 AI Stock Analyzer\n"
        "**Portfolio-Aware | Risk Scenarios | Cash Management | Multi-Agent AI**\n"
        "*8 stages | Gemini 2.5 Flash + Claude Sonnet 4.5 | ~30-45s*"
    )

    with gr.Tabs():
        with gr.Tab("🔍 Full Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Input & Settings")
                    ti = gr.Textbox(
                        label="Ticker Symbol",
                        placeholder="AAPL, NVDA, SPY...",
                    )
                    with gr.Accordion("💼 Portfolio Context (Optional)", open=False):
                        gr.Markdown(
                            "Enter your current holdings and cash position. Format: `TICKER:PCT%`.\n"
                            "Example: `AAPL:15%, NVDA:10%, MSFT:12%, CASH:20%`"
                        )
                        pi = gr.Textbox(
                            label="Current Portfolio",
                            placeholder="AAPL:15%, NVDA:10%, MSFT:12%, CASH:20%",
                            lines=3,
                        )
                        target_alloc = gr.Textbox(
                            label="Target Allocation % for New Ticker",
                            placeholder="e.g. 5",
                        )
                    ab = gr.Button("🚀 Run Analysis", variant="primary")
                    so = gr.Markdown("")
                
                with gr.Column(scale=3):
                    ro = gr.HTML("")
                    
            ab.click(fn=run_full, inputs=[ti, pi, target_alloc], outputs=[so, ro])
            ti.submit(fn=run_full, inputs=[ti, pi, target_alloc], outputs=[so, ro])

        with gr.Tab("💼 Portfolio Health"):
            gr.Markdown(
                "**Portfolio Health Check + AI Recommendations**\n"
                "Scores your current holdings, analyzes sector exposure, recommends cash targets, "
                "and uses Claude to suggest complementary tickers."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    port_input = gr.Textbox(
                        label="Current Portfolio",
                        placeholder="AAPL:15%, NVDA:10%, MSFT:12%, SGOV:20%",
                        lines=5,
                    )
                    port_btn = gr.Button("📊 Review Portfolio", variant="primary")
                with gr.Column(scale=3):
                    port_output = gr.Textbox(label="Portfolio Review Results", lines=30, interactive=False)
            
            port_btn.click(fn=run_portfolio_review, inputs=[port_input], outputs=[port_output])

        with gr.Tab("Guide"):
            gr.Markdown("""## Pipeline
| # | Role | Model | Cost |
|---|------|-------|------|
| 1 | Data + ATR + Chart + Backtest + Sector RS | Local | Free |
| 1b | Portfolio enrichment + correlations + risk scenarios | Local + yfinance | Free |
| 1c | SEC Filing data (10-K/10-Q financials + MD&A + Risk Factors) | edgartools + SEC EDGAR | Free |
| 2-5 | Research, Macro, Earnings, Insider (PARALLEL) | Gemini 2.5 Flash | Free* |
| 5b | Social Sentiment (X/Twitter) | Grok 3 Mini Fast | ~/usr/bin/bash.01 |
| 6 | Trader (portfolio-aware) | Claude Sonnet 4.5 | ~$0.04 |
| 7 | Contrarian + Fatal Flaw | Gemini 2.5 Flash | Free* |
| 8 | Final PM (portfolio + cash + risk aware) | Claude Sonnet 4.5 | ~$0.04 |

*Free within daily grounding quota (500 RPD). Cached: repeat analyses same day = $0.

## ATR Position Sizing
| Daily ATR% | Max Position Size |
|------------|-------------------|
| < 2% | 3-5% of portfolio |
| 2-3% | 2-4% of portfolio |
| 3-4% | 1-3% of portfolio |
| > 4% | Max 1-2% of portfolio |

## Cash Target by Market Regime
| Fear & Greed | Recommended Cash | Regime |
|-------------|-----------------|--------|
| 0-20 (Extreme Fear) | 5% | Deploy aggressively (contrarian) |
| 20-35 (Fear) | 10% | Cautious accumulation |
| 35-55 (Neutral) | 15% | Standard buffer |
| 55-75 (Greed) | 20% | Raise cash, overextended |
| 75-100 (Extreme Greed) | 30% | Maximum defensiveness |

*Adjusted for portfolio beta and sector concentration.*

## New Features
- **Portfolio-Aware Analysis**: Enter holdings, get sizing relative to existing exposure
- **Cash Management**: Dynamic cash target based on F&G, beta, sector concentration
- **Trim Recommendations**: Suggests which positions to trim to meet cash target
- **Risk Scenario Module**: Stress tests with beta-adjusted drawdowns
- **Gemini Research Cache**: 24hr cache saves grounding costs on repeat analyses
- **Sector Relative Strength**: Compares stock vs sector ETF over 20/60/120 days
- **Earnings Proximity Warning**: Flags earnings within 14 days
- **OBV Chart Panel**: On-Balance Volume for volume confirmation
- **Portfolio Tab**: Health check on holdings, sector gap analysis, AI-recommended additions
- **Correlation Warnings**: Flags highly correlated pairs in portfolio and recommendations
- **Social Sentiment**: Live X/Twitter sentiment analysis via Grok — retail chatter, FinTwit narratives, sentiment divergence detection
- **SEC Filing Integration**: Multi-year XBRL financials + MD&A narrative + Risk Factors from 10-K/10-Q (via edgartools, free, no API key)

*Educational purposes only. Not financial advice.*""")

print("\n🚀 AI Stock Analyzer")
print("Launching Gradio interface...\n")
app.launch(share=False, debug=False, auth=(os.environ.get("GRADIO_USERNAME", "hkim"), os.environ.get("GRADIO_PASSWORD", "test123")))



🚀 AI Stock Analyzer
Launching Gradio interface...

* Running on local URL:  http://127.0.0.1:7860


01:16:16 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
01:16:16 [INFO] HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
01:16:16 [INFO] HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
01:16:16 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
01:16:17 [INFO] HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://31e8317ae41d21a375.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


01:16:19 [INFO] HTTP Request: HEAD https://31e8317ae41d21a375.gradio.live "HTTP/1.1 200 OK"


01:16:19 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
